# RSNA Knee: deterministic report supervision + MRI attention

**Training and offline inference in one self-contained notebook. No LLM pseudo-labels.**

The supplied E13 diagram becomes: training reports → multilingual deterministic clauses,
negation and severity rules → masked soft targets, overridden by available official expert
labels → six MRI sequence slots → self-supervised image encoder → twelve cross-series
finding queries → grouped folds and fixed-weight rank ensemble → `submission.csv`.
Test predictions use **MRI only**. No test reports, hardcoded study predictions, external
label CSVs, text-generation API, or pretrained competition heads are consumed.

The default member is DINOv2 ViT-S/14. DINOv3 ViT-S/16, compatible RadImageNet ResNet-50
foundation weights, and finding specialists are configurable below. A missing optional
member is never silently replaced with another model. This is a new pipeline, not a
reproduction of E13's score.

## Kaggle run sequence

1. Add the **RSNA Knee Abnormality Detection** competition dataset; accept its rules.
   Select a GPU accelerator. First run with `RUN_MODE = "train"` and Internet **on**,
   or supply compatible local foundation weights and offline dependency wheels.
2. Run all cells. The first run extracts MRI features once and trains five grouped heads.
   The default `finetune_epochs=0` is the efficient frozen-encoder baseline. Setting it
   to 2 adds late-block fine-tuning for every fold; use a **new output directory** when
   changing an experiment. Full three-member training is several runs, not a promised
   single Kaggle GPU session. The notebook supports one GPU and does not assume two.
3. Save/download the output directory `rsna_artifacts`. Publish its `manifest.json` and
   all listed `.pt` files as your own Kaggle model/dataset. Image and feature caches,
   target CSVs, and OOF diagnostics are useful research artifacts but aren't needed
   as inference inputs. Do not change weights or the manifest after training.
4. For a submission copy, set `RUN_MODE = "infer"`, point `CHECKPOINT_DIR` at the attached
   directory containing `manifest.json`, and disable Internet. Reuse this exact generated
   implementation. Execute all cells. The output is `/kaggle/working/submission.csv`.
   The hidden test IDs are discovered dynamically from `test.csv`. No training CSV is
   read in inference. If a dependency is missing, attach a compatible wheel dataset
   and set `WHEEL_DIR`; inference will not install from the Internet.

`train_infer` is available for a training-time smoke submission. A training commit with
Internet on is not the scored offline inference notebook. A three-study example run
does not establish hidden-set runtime or leaderboard performance.

## Evaluation contract

Official labels can be missing independently per target. Each observed expert label
overrides that target's rule score. Unmentioned/uncertain rule labels receive zero
loss weight. Expert-label weights and member weights are fixed before evaluation.
Groups join exact normalized duplicate reports and available patient IDs. When no
patient ID is supplied, patient-disjointness cannot be claimed. Rules are not fit
against the 58 gold studies, and the synovitis proxy backoff in E13 is disabled.

An inner grouped split chooses head epochs; then the head is refit on all outer training
groups. Optional fine-tuning uses a fixed epoch count. Outer-fold labels never select
epochs, weights, or thresholds. `validation.json` reports gold-only, per-target OOF
diagnostics and undefined AUCs; pooled fold ranks are not a Kaggle leaderboard score.
Rank blending does not threshold predictions. Specialists contribute only to explicitly
named targets; no unvalidated manual test-time rule adjustment is applied.

## Public solution research — checked 13 October 2026

Direct Kaggle browsing was blocked by the development environment's network policy.
I searched accessible public GitHub repositories, including recently updated entries,
and inspected their documents. This is not an exhaustive ranking of current Kaggle code.

| Public source | Attributed result | Use here |
|---|---|---|
| [NTejas-1 reproducible pipeline](https://github.com/NTejas-1/RSNA-Knee-Abnormality-Detection), repository timestamp 27 Sep 2026 | README reports public **0.941** and parent **0.940** | Sequence slots, attention, and rank blending informed design; its LLM labels and competition checkpoints are excluded. |
| [Bend the Knee to Speedy Raptors](https://www.kaggle.com/code/mattiaangeli/bend-the-knee-to-speedy-raptors-the-original) | **0.940**, attributed by the preceding README; page not directly accessible | Reference link only; no imported artifacts. |
| [Yashjamine1207 own-model pipeline](https://github.com/Yashjamine1207/RSNA-Knee-Abnormality-Detection), updated 12 Oct 2026 | README reports public **0.930** (ensemble), **0.921** (first arm) | It uses LLM-derived report tables, which are excluded. |
| [Alvaro Mendizabal research system](https://github.com/alvaromendizabal/rsna-knee-abnormality-detection), updated 13 Oct 2026 | README reports historical external **0.943**; not verified as a public notebook leaderboard score | Evidence for grouped validation and engineering practices; no performance claim is transferred. |
| [Competition documentation mirror](https://github.com/AhmadEnan/RSNA-Knee-Abnormality-Detection/blob/main/COMPETITION.md) | Documents 12-column macro ROC-AUC and offline notebook inference | Cross-checked the data/schema contract against E13; check Kaggle for current rules. |

These scores are author-reported, not independently verified. No measured competition
score exists for this new notebook. Removing LLM labels changes the supervision, so
the referenced results are not expected scores for this implementation.


In [ ]:
# Settings read before dependency installation. Change these for your run.
RUN_MODE = "infer"  # "train", "infer", "train_infer"
CHECKPOINT_DIR = "/kaggle/input/datasets/notfu4y/rsna-artifacts/rsna_artifacts/"  # for now path is /kaggle/input/datasets/notfu4y/rsna-artifacts
WHEEL_DIR = ""  # attached offline wheels when needed
INSTALL_MISSING = True

import importlib.util
import importlib.metadata
import subprocess
import sys
from pathlib import Path

# Kaggle supplies CUDA-enabled torch/torchvision. Do not replace them with CPU wheels.
for module in ("torch", "torchvision"):
    if importlib.util.find_spec(module) is None:
        raise RuntimeError("Select Kaggle's PyTorch GPU image; torch/torchvision is missing")

requirements = {
    "numpy": "numpy==2.2.6", "pandas": "pandas==2.2.3",
    "sklearn": "scikit-learn==1.6.1", "pydicom": "pydicom==3.0.1",
    "PIL": "Pillow==11.1.0", "timm": "timm==1.0.20",
    "huggingface_hub": "huggingface-hub>=0.34,<3",
    "safetensors": "safetensors>=0.4", "yaml": "PyYAML>=6",
}
missing = [package for module, package in requirements.items()
           if importlib.util.find_spec(module) is None]
if importlib.util.find_spec("timm") is not None:
    from packaging.version import Version
    if Version(importlib.metadata.version("timm")) < Version("1.0.20"):
        missing.append("timm==1.0.20")
if missing:
    if not INSTALL_MISSING:
        raise RuntimeError("Missing dependencies: " + ", ".join(missing))
    arguments = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-cache-dir"]
    if WHEEL_DIR:
        if not Path(WHEEL_DIR).is_dir():
            raise FileNotFoundError(WHEEL_DIR)
        arguments += ["--no-index", "--find-links", WHEEL_DIR]
    elif RUN_MODE == "infer":
        raise RuntimeError("Offline inference needs an attached wheel directory for: " + ", ".join(missing))
    # Install only absent packages. Kaggle's existing CUDA torch stays installed.
    subprocess.run(arguments + missing, check=True)
print("Dependencies available; mode:", RUN_MODE)


## Deterministic multilingual E13 lexicon

These are fixed regex rules, not a language model. The training pipeline masks silence.

In [ ]:
# SPDX-License-Identifier: Apache-2.0
# Deterministic report lexicon adapted from the user-supplied E13 notebook.
# Original notebook credits prvsiyan, pilkwang, stevenleehans, lixin73,
# tonylica, marwanmath, cf696666, romantamrazov and Mattia Angeli.
# Original: https://www.kaggle.com/code/prvsiyan/rsna-knee-read-the-report-then-the-knee
# Changes: isolated the deterministic lexicon; disabled synovitis proxy backoff.
# No external label tables or language models are loaded here.
from __future__ import annotations
import re
import unicodedata
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
_PRE = str.maketrans({'ı': 'i', 'İ': 'i', 'I': 'i', 'ß': 'ss', 'đ': 'd', 'Đ': 'd', 'ø': 'o', 'Ø': 'o', 'æ': 'ae', 'Æ': 'ae'})

def normalize(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.translate(_PRE).lower()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join((ch for ch in text if not unicodedata.combining(ch)))
    text = text.replace('\xad', '')
    text = re.sub('[_\\-/\\\\]+', ' ', text)
    text = re.sub('[ \\t]+', ' ', text)
    return text
_SENT_SPLIT = re.compile('(?<=[.;!?])\\s+|\\n+')

def unwrap(text: str) -> str:
    if not isinstance(text, str):
        return ''
    out = []
    for line in text.split('\n'):
        s = line.strip()
        if out and out[-1] and (not re.search('[.;:!?>*•]$', out[-1])) and (len(out[-1].split()) >= 4) and s and (not s[:1].isupper()):
            out[-1] = out[-1] + ' ' + s
        else:
            out.append(s)
    return '\n'.join(out)

def clauses(text: str):
    norm = normalize(unwrap(text) if FEATURES['unwrap'] else text)
    raw = [c.strip() for c in _SENT_SPLIT.split(norm) if c and c.strip()]
    merged = []
    for i, c in enumerate(raw):
        if c.endswith(':') and len(c.split()) <= 14 and (i + 1 < len(raw)):
            merged.append(c + ' ' + raw[i + 1])
        merged.append(c)
    out = []
    for c in merged:
        out.append(c)
        if len(c.split()) > 25:
            out.extend((p.strip() for p in c.split(',') if len(p.split()) > 2))
    return out
FEATURES = {'unwrap': True, 'directional_negation': True, 'oa_inherit': True, 'graded_pathology': True, 'synovitis_backoff': True}

def _rx(*alts: str) -> re.Pattern:
    return re.compile('|'.join(alts))
PRE_NEG = _rx('\\bno\\b', '\\bnot\\b', '\\bwithout\\b', '\\bnegative for\\b', '\\babsence\\b', '\\bno evidence\\b', '\\bfree of\\b', '\\bnone\\b', '\\bneither\\b', '\\bnor\\b', '\\bsin\\b', '\\bno hay\\b', '\\bausencia\\b', '\\bausentes?\\b', '\\bno se\\b', '\\bpas de\\b', '\\bsans\\b', '\\baucune?\\b', '\\bgeen\\b', '\\bzonder\\b', '\\bniet\\b', '\\bkeine?[nmrs]?\\b', '\\bohne\\b', '\\bnicht\\b', '\\bkein\\b', '\\bnema\\b', '\\bbez\\b', '\\bnisu\\b', '\\bnije\\b', '\\bδεν\\b', '\\bχωρις\\b', 'ουδεν', '\\bουτε\\b', '\\bбез\\b', '\\bне\\b', 'липсва', '\\bняма\\b')
POST_NEG = _rx('\\byok\\b', '\\byoktur\\b', 'izlenmemekte', 'saptanmadi', '\\bdegil\\b', 'gozlenmemekte', 'mevcut degil', 'eslik etmiyor', '\\bizlenmedi\\b', 'izlenmemistir', 'saptanmamistir', 'gorulmemistir', '\\bnema znakova\\b', 'bez znakova')
NEGATION = _rx(PRE_NEG.pattern, POST_NEG.pattern, '\\bunremarkable\\b')
NEG_WINDOW = 90

def _negated(clause: str, start: int, end: int) -> bool:
    for m in PRE_NEG.finditer(clause):
        if m.end() <= start and start - m.end() <= NEG_WINDOW:
            if not re.search('\\b(but|however|ancak|fakat|pero|maar|aber|no i|ali|ωστοσο|αλλα|но)\\b', clause[m.end():start]):
                return True
    for m in POST_NEG.finditer(clause):
        if m.start() >= end and m.start() - end <= NEG_WINDOW:
            return True
    return False
NORMALITY = _rx('\\bnormal', '\\bintact\\b', '\\bpreserved\\b', '\\bwithin normal limits\\b', 'limites normales', '\\bconservad', '\\bintegr', '\\bnormales\\b', '\\bdoga(l|ll)\\b', 'korunmus', '\\bnormaldir\\b', 'olagan', '\\buredn', '\\bocuvan', '\\bodrzan', '\\bintakt', '\\bprimjeren', '\\bodrzanog kontinuiteta', '\\bodržan', 'φυσιολογικ', 'ακεραι', 'δεν παρατηρουνται', 'δεν σημειωνονται', 'unauffallig', 'regelrecht', '\\bo\\.?b\\.?\\b', 'нормал', 'запазен', 'съхранен', '\\bбез особености\\b', 'интактн', '\\bgaaf\\b', '\\bnormaal\\b')
NORMAL_PHRASE = _rx('\\bsin alteracion', '\\bsin cambios\\b', '\\bsin particularidad', '\\bsin hallazgos\\b', '\\bsin lesion', '\\bsin signos de (rotura|lesion)', '\\bcontinu[oa]s?\\b', '\\bcontinuidad conservada\\b', '\\bno abnormalit', '\\bno significant abnormalit', '\\bunremarkable\\b', '\\bno evidence of (tear|injury|abnormalit)', '\\bohne auffalligkeit', '\\bkein nachweis\\b', '\\bohne befund\\b', '\\bgeen afwijking', '\\bzonder afwijking', '\\bsans anomalie', "\\bpas d[e']anomalie", '\\bbez osobitosti\\b', '\\bbez znakova (rupture|lezije)\\b', '\\bbez patoloskih\\b', 'χωρις αλλοιωσ', 'χωρις παθολογ', 'δεν παρατηρουνται (αξιολογα|παθολογ)', '\\bбез особености\\b', '\\bбез патологич', '\\bбез данни за\\b', '\\bozel bir ozellik yok', '\\bpatolojik bulgu (yok|izlenmemis)')
UNCERTAIN = _rx('\\bpossible\\b', '\\bprobable\\b', '\\bsuspicious\\b', '\\bsuspected?\\b', 'cannot (be )?exclude', '\\bmay\\b', '\\bquestionable\\b', '\\bequivocal\\b', '\\br/o\\b', '\\bdd\\b', '\\blikely\\b', '\\bsuggest', '\\bcompatible with\\b', '\\bposible\\b', 'sin criterios categoricos', '\\bdudos', '\\bsugier', '\\bmuhtemel\\b', '\\bolasi\\b', '\\bsupheli\\b', '\\bizlenim', '\\bdusundur', '\\bmoguce\\b', '\\bvjerojatno\\b', '\\bsumnja\\b', '\\bmoze odgovarati\\b', 'πιθαν', 'υποπτ', '\\bmoglich', '\\bverdachtig', '\\bfraglich', '\\bv\\.?a\\.?\\b', '\\bwohl\\b', '\\bвъзможно\\b', '\\bвероятно\\b', 'суспект', '\\bmogelijk\\b', '\\bverdacht\\b')

TEAR = _rx('\\btear', '\\btorn\\b', '\\brupture', '\\bdisruption\\b', 'discontinuit', '\\bavuls', '\\bmacerat', '\\bbuckethandle\\b', 'bucket handle', '\\brotura\\b', '\\broturas\\b', '\\bruptura', '\\bdesgarro', '\\broto\\b', '\\bdechirure', '\\bdechire', '\\bscheur', '\\bruptuur', 'gescheurd', '\\briss\\b', 'einriss', '\\bruptur', 'zerreiss', '\\blasion', '\\bausriss', '\\byirtik', '\\byirtig', '\\bkopma\\b', 'butunluk kaybi', '\\brupturu\\b', 'devamsizlik', '\\brupture\\b', '\\bdevamliligi secilememis', '\\bpuknuce', '\\bprekid\\b', '\\bpukotin', '\\bruptur', 'ρηξη', 'ρηξις', 'ρηγμα', 'ασυνεχεια', 'руптура', 'разкъсв', 'разрив', 'скъсв', '\\bлезия\\b')
DEGEN = _rx('degenerat', '\\bmucoid\\b', '\\bmyxoid\\b', '\\bfray', '\\bfissur', 'dejeneratif', '\\bmukoid\\b', 'degenerativn', 'εκφυλ', 'дегенерат', '\\bμυξοειδ', '\\bμυξωδ', '\\bmeniskopat', '\\bmeniscopath', '\\bmuco ?ide\\b', 'aufgefasert', '\\bdejenerasyon\\b')
INJURY = _rx('\\binjur', '\\bsprain', '\\blesion', '\\blasion', '\\bedema\\b', '\\boedema\\b', '\\bodem\\b', '\\bedem\\b', '\\bοιδημα', '\\bодем', '\\bедем', '\\bstrain\\b', '\\bhigh signal\\b', '\\bsignal alteration\\b', '\\bhiperintens', '\\bhyperintens', 'aumento de senal', 'alteracion de senal', 'cambio de senal', '\\bsignalanhebung', '\\bsignalalteration', 'verhoogd signaal', 'sinyal artis', 'αυξημενο σημα', 'повишен сигнал', '\\besguince\\b', '\\bthicken', '\\bzadebljanje\\b', '\\bverdikking\\b', '\\bdistenzij', '\\blaksite\\b', '\\blaxity\\b', '\\bpartial\\b', '\\bparcijaln', '\\bparcial', '\\bpartiel', '\\bpartiell')
_GRADE_RX = re.compile('(?:grade|grad|grado|grau|derece|stupnja|stupanj|βαθμ|степен|icrs|outerbridge)[\\s:]*(?:grade\\s*)?([1-4]|iv|iii|ii|i)\\b')
_ROMAN = {'i': 1, 'ii': 2, 'iii': 3, 'iv': 4}

def _grade_of(clause: str):
    best = None
    for m in _GRADE_RX.finditer(clause):
        v = m.group(1)
        n = _ROMAN.get(v, None) if not v.isdigit() else int(v)
        if n is not None and (best is None or n > best):
            best = n
    return best
ANAT = {'ACL': _rx('anterior cruciate', '\\bacl\\b', 'cruzado anterior', '\\blca\\b', 'croise anterieur', 'voorste kruisband', '\\bvkb\\b', 'vorderes kreuzband', 'vorderen kreuzband', 'vordere kreuzband', 'on capraz', '\\bocb\\b', 'anterior capraz', 'prednji krizni', 'prednjeg krizn', 'προσθι[οα][^ ]* χιαστ', 'προσθιου χιαστου', 'χιαστο[^ ]* συνδεσμ', '\\bχιαστ\\w*', 'предна кръстна', 'предната кръстна', 'предна кръста', 'cruciate ligaments', 'ligamentos cruzados', 'ligaments croises', 'kruisbanden', 'kreuzbander', 'capraz baglar', 'krizn[a-z]* ligament[a-z]*', 'χιαστοι συνδεσμ', 'χιαστων συνδεσμ', 'кръстните връзки', 'кръстни връзки'), 'MCL': _rx('medial collateral', '\\bmcl\\b', 'tibial collateral', 'colateral medial', 'colateral interno', '\\blcm\\b', 'collateral medial', 'collateral interne', 'mediale collaterale', 'binnenband', '\\b(mediale|laterale) banden\\b', '\\bcollaterale banden\\b', 'innenband', 'mediales? kollateral', '\\bic yan bag', 'medial kollateral', '\\biyb\\b', 'medyal kollateral', 'medijalni kolateraln', 'medijalnog kolateraln', 'εσω πλαγι', 'εσωτερικο πλαγι', '\\bπλαγι\\w* συνδεσμ', '\\bπλαγιοι\\b', 'медиален колатерал', 'вътрешна странична', '\\bколатерал\\w*', '\\bcolaterales\\b', '\\bcollateraux\\b', '\\bcollateralen\\b', '\\bkolateralni\\b', 'collateral ligaments', 'ligamentos colaterales', 'ligaments collateraux', 'collaterale banden', 'kollateralbander', 'seitenbander', 'yan baglar', 'kolateraln[a-z]* ligament[a-z]*', 'πλαγιοι συνδεσμ', 'πλαγιων συνδεσμ', 'колатерални връзки', 'страничните връзки'), 'Medial Meniscus': _rx('medial meniscus', '\\bmm\\b(?= tear)', 'medial menisc', 'menisco medial', 'menisco interno', 'menisque medial', 'menisque interne', 'mediale meniscus', 'binnenmeniscus', 'innenmeniskus', 'medialen? meniskus', 'innenmeniskushinterhorn', 'medyal menisk', '\\bic menisk', 'medijalni meniskus', 'medijalnog meniskusa', 'medijalnom meniskusu', 'medijaln\\w* menisk\\w*', '\\bmedijalnog meniska\\b', 'medijalni menisk', 'εσω μηνισκ', 'μηνισκ[^ ]* του εσω', 'εσω διαμερισμα[^.]{0,40}μηνισκ', 'медиалния менискус', 'медиален менискус', 'вътрешния менискус', 'oba meniska', 'both menisci', 'ambos meniscos', 'beide menisci', 'her iki menisku', 'amfoteroi\\w* mhnisk', 'αμφοτερ\\w* μηνισκ', 'двата менискуса', 'medial (and|&) lateral menisc'), 'Lateral Meniscus': _rx('lateral meniscus', 'lateral menisc', 'menisco lateral', 'menisco externo', 'menisque lateral', 'menisque externe', 'laterale meniscus', 'buitenmeniscus', 'aussenmeniskus', 'lateralen? meniskus', 'aussenmeniskushinterhorn', 'lateral menisk', '\\bdis menisk', 'lateralni meniskus', 'lateralnog meniskusa', 'lateralnom meniskusu', 'lateraln\\w* menisk\\w*', '\\blateralnog meniska\\b', 'εξω μηνισκ', 'μηνισκ[^ ]* του εξω', 'εξω διαμερισμα[^.]{0,40}μηνισκ', 'латералния менискус', 'латерален менискус', 'външния менискус', 'oba meniska', 'both menisci', 'ambos meniscos', 'beide menisci', 'her iki menisku', 'αμφοτερ\\w* μηνισκ', 'двата менискуса', 'medial (and|&) lateral menisc')}
OA_EVIDENCE = _rx('osteoarthrit', '\\barthros', '\\bgonarthros', '\\bosteoarthros', 'chondropath', 'chondromalac', 'condropat', 'condromalac', '\\bchondros', '\\bchondrosis\\b', 'chondral (loss|defect|ulcer|thinning|injury|fissur|wear)', 'cartilage (loss|thinning|defect|fissur|wear|damage|heterogeneity|irregularit)', '(loss|thinning|fissur|defect|ulcer|erosion|denudation) of[^.]{0,20}cartilage', 'articular cartilage[^.]{0,30}(loss|thin|fissur|defect|erosion|wear|irregular)', 'osteophyt', 'osteofit', 'osteofyt', 'osteofito', 'osteophyten', 'spurring', 'joint space narrowing', 'pinzamiento articular', 'reduced joint space', 'kikirdak kayb', 'kikirdak incelme', 'kondropati', 'kondral', 'kikirdak dejener', 'eklem aralig\\w* daral', 'eklem mesafesi daral', 'kikirdak kalinlig\\w* azal', 'kraakbeen', 'gonartrose', 'artrose', '\\bknorpel', 'arthrose', 'gonarthrose', 'hrskavic', 'hondromalac', 'artroz', 'osteoartrit', 'artrotsk', 'artrotick', '\\boa promjen', '\\boa\\b', 'degenerativne promjene hrskav', 'χονδρ[^ ]*παθ', 'αρθριτ', 'αρθρωσ', 'οστεοφυτ', 'χονδρομαλακ', 'αρθρικου χονδρου', 'εξαλειψη του αρθρικου χονδρου', 'διαβρωση του αρθρικου χονδρ', 'λεπτυνση[^.]{0,30}χονδρ', 'φθορα[^.]{0,20}χονδρ', 'артроз', 'хондропат', 'остеофит', 'хрущял[^.]{0,40}(изтън|увред|дефект|липс)', 'изтъняване[^.]{0,30}хрущял', 'хондромалац', 'ulcera[s]? condral', 'cartilago[^.]{0,25}(perdida|adelgaz)', 'icrs grade', 'icrs\\b', 'outerbridge', '\\bdenudation\\b', 'denudacij', 'erozivne promjene', '\\berosion of[^.]{0,20}cartilage', 'kraakbeenlijden', 'kraakbeenverlies')
TF_SITE = _rx('compartment', 'compartimento', 'compartiment', 'kompartman', 'kompartiment', 'kompartment', 'odjelj', 'διαμερισμα', 'компартм', '\\bотдел', 'femorotibial', 'tibiofemoral', 'femoro tibial', 'femorotibiaal', 'femorotibijaln', 'феморотибиал', '\\bft zglob', 'tibiofemoraln', 'condyle', 'condilo', 'kondyl', 'kondil', 'condyl', 'κονδυλ', 'кондил', '\\bplateau', '\\bplato\\b', 'platillo', 'meseta', 'плато', 'tibiaplateau', 'tibijaln\\w* plato', 'tibyal plato', 'tibia plato', 'κνημιαι', 'μηριαι', 'weightbearing', 'weightbaring', 'zona de carga', 'dragende deel', 'agirlik tasiyan', '\\bfemur\\b', '\\btibia\\b', '\\bfemoral\\b', '\\btibial\\b', '\\bfemura\\b', '\\btibije\\b', '\\bmesarthrio\\b', 'μεσαρθριο')
PF_SITE = _rx('patellofemoral', 'femoropatellar', 'femoropatelar', 'patelofemoral', 'retropatellar', 'retrorotulian', 'trochlea', 'troclea', 'troklea', 'trochlear', 'trohlej', 'τροχιλ', '\\bpatella', '\\bpatellar', 'rotulian', '\\brotula\\b', '\\bpatele\\b', 'patellofemoraal', 'femoropatellair', 'επιγονατιδ', 'μηροεπιγονατιδ', 'пател', 'феморопател', 'anterior compartment', 'compartimento anterior', 'prednj\\w* odjeljk', '\\bfp zglob', '\\bpf zglob', '\\bfaset', '\\bfacet', 'patellofemoraln')
SIDE_MEDIAL = _rx('\\bmedial\\w*', '\\bmedyal\\w*', '\\bmedijaln\\w*', '\\bmediaal\\w*', '\\bmediale\\w*', '\\binterno\\b', '\\binterna\\b', '\\binternos\\b', '\\binterne\\b', '\\binnen\\w*', '\\bic\\b', '\\bunutarnj\\w*', '\\bεσω\\w*', '\\bεσωτερικ\\w*', '\\bмедиал\\w*', '\\bвътреш\\w*', '\\bbinnen\\w*', '\\bmediaal\\b', '\\bmediales?\\b')
SIDE_LATERAL = _rx('\\blateral\\w*', '\\bexterno\\b', '\\bexterna\\b', '\\bexternos\\b', '\\bexterne\\b', '\\bdis\\b', '\\blateraln\\w*', '\\baussen\\w*', '\\bbuiten\\w*', '\\bεξω\\w*', '\\bεξωτερικ\\w*', '\\bлатерал\\w*', '\\bвъншн\\w*', '\\bvanjsk\\w*')
SIDE_ANTERIOR = _rx('\\banterior\\w*', '\\bant\\b', '\\bon\\b', '\\bprednj\\w*', '\\bvorder\\w*', '\\bvoorste\\b', '\\bπροσθι\\w*', '\\bпредн\\w*', '\\banteriyor\\w*', '\\bavant\\b', '\\banterieur\\w*')
GLOBAL_OA = _rx('tri ?compartment', 'all three compartment', 'global(ised)? (oa|osteoarthrit)', '\\bgonarthros', '\\bgonartros', '\\bgonarthrose', '\\bgonartrose', 'gonartro', 'goanrtrot', 'gonartrot', 'osteoarthritis of the knee', 'artrosis (de |)(la )?rodilla', 'knee osteoarthrit', '\\bdiz osteoartrit', '\\bgonartroz', 'artroza koljena', 'οστεοαρθριτιδα', 'αρθριτιδα του γονατος', 'εκφυλιστικη οστεοαρθριτ', 'артроза на колянната', 'гонартроз', 'degenerative joint disease', '\\bdjd\\b', 'three compartments', 'compartmens', 'compartments')
DIRECT = {'Effusion': _rx('\\beffusion', 'joint fluid', 'intra ?articular fluid', '\\bhydrops\\b', '\\bhemarthros', '\\bhaemarthros', 'derrame articular', '\\bderrame\\b', 'liquido articular', 'hemartrosis', 'epanchement', 'gewrichtsvocht', '\\bvocht\\b', 'gewrichtseffusie', 'opzetting van suprapatell', 'gelenkerguss', '\\berguss\\b', 'gelenksergu', 'gelenksflussigkeit', 'eklem\\w* ic\\w* sivi', 'efuzyon', 'eklem sivisi', 'eklem mesafesinde sivi', 'sivi (miktari|artisi|birikimi)', 'sivi artis', '\\bsivi\\b[^.]{0,25}artmis', '\\bizljev', '\\bizliv', 'zglobn[^ ]* tekucin', '\\bhidrops\\b', 'αρθρικ[^ ]* υγρ', 'υγρου ενδαρθρικα', 'ενδαρθρικ[^ ]* υγρ', 'ποσοτητα υγρου', 'ενδαρθρικ', 'αρθρικη συλλογη', 'υγρο στην αρθρωση', 'υγρου στην αρθρωση', 'συλλογη υγρου', 'ενθαρθρικ', 'ставен излив', 'излив', 'ставна течност', 'синовиална течност'), 'Synovitis': _rx('synovit', 'sinovit', 'synovial (thickening|proliferation|hypertroph)', 'thicken\\w* synovial', 'hypertroph\\w* of the synovium', 'synoviale? (verdikking|proliferatie)', 'verdikkingen van (het )?synovium', 'synovialitis', 'synovialis(verdickung|proliferation)', 'reizsynovial', 'sinovijalitis', 'sinovitis', 'zadebljanje sinovij', 'proliferacij\\w* sinovij', 'sinovijaln\\w* proliferacij', 'υμενιτιδα', 'συνοβιτιδα', 'υμενικ[^ ]* υπερτροφ', 'αρθρικου υμεν', 'παχυνση[^.]{0,20}υμεν', 'υμενα', 'синовит', 'синовиал[^ ]* (задебел|пролифер)', '\\bpannus\\b', '\\bhoffit', 'sinovyal\\w* (kalinlas|proliferas)', 'sinovyal hipertrof', '\\bartrit\\b', '\\barthritis\\b'), "Baker's": _rx('baker', 'popliteal cyst', 'quiste popliteo', 'quistes popliteos', 'kyste poplite', 'popliteale? cyst', 'poplitealzyste', 'bakerzyste', 'popliteal kist', '\\bbakerova\\b', 'poplitealn[^ ]* cist', 'popliteal\\w* cist', 'κυστη baker', 'πολυχωρη συνοβιακη κυστη', 'κυστη του baker', 'συνοβιακη κυστη', 'κυστη τυπου baker', 'киста на бейкър', 'бейкърова киста', 'поплитеална киста', 'бекеров', 'gastrocnemio ?semimembranos', 'gastrocnemius semimembranosus burs'), 'Contusion': _rx('\\bcontusion', 'bone bruise', 'bone marrow (o?edema|contusion)', 'marrow o?edema', '\\bkontuz', 'medular bone o?edema', 'osseous contusion', 'contusion osea', 'edema oseo', 'edema de medula osea', 'contusiones oseas', 'oedeme osseux', 'contusion osseuse', 'botcontusie', 'botoedeem', 'beenmergoedeem', 'botmergoedeem', 'knochenmarkodem', 'knochenodem', 'knochenmarksodem', 'kontusion', 'kemik kontuzyonu', 'kemik iligi odemi', 'kemik odemi', 'kemik iliginde odem', 'kontuzyonel kemik', 'kemik iligi odemleri', 'kostani edem', 'edem kosti', 'kontuzij', 'kostane srzi[^.]{0,20}edem', 'οστεομυελικ[^ ]* οιδημα', 'οστικο οιδημα', 'μυελικο οιδημα', 'οστικο μωλωπ', 'костномозъчен едем', 'костен едем', 'контузионен', 'костно мозъчен едем'), 'Fracture': _rx('\\bfractur', '\\bfract\\b', '\\bfractura', '\\bfracturas\\b', '\\bfractuur', '\\bbreuk\\b', '\\bfraktur', '\\bbruch\\b', '\\bkirik\\b', '\\bkirigi\\b', '\\bkiri[kg]\\w*', '\\bprijelom', 'impresijsk[^ ]* fraktur', 'impaktcij', 'καταγμα', 'καταγματ', 'фрактур', 'счупван', 'фисур', 'insufficiency fracture', 'stress fracture', 'avulsion fracture', 'subchondral fracture', 'subkondral kiri', 'impaction (fracture|injury)', 'osteochondral (fracture|impaction)', '\\bsegond\\b', 'impactiefractuur', 'subchondrale impression', 'subchondraler? impress')}
DECOY = {'Fracture': _rx('microfractur', '\\bfracture (risk|prophyla)'), "Baker's": _rx('meniscal cyst', 'quiste meniscal', 'parameniscal')}
PAIRED = {'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus'}
OA_TARGETS = ['Medial OA', 'Lateral OA', 'PF OA']
PLURAL_MENISCI = _rx('\\bmenisci\\b', '\\bmeniscos\\b', '\\bmenisques\\b', '\\bmenisken\\b', '\\bmeniskusi\\b', '\\bmenisk\\w*ler\\b', '\\bμηνισκοι\\b', '\\bμηνισκων\\b', '\\bменискуси\\b', '\\bменискусите\\b', '\\bmenisci\\w*\\b')
ANY_SIDE = _rx(SIDE_MEDIAL.pattern, SIDE_LATERAL.pattern)
STEM_MENISCUS = _rx('menisc\\w*', 'menisk\\w*', 'μηνισκ\\w*', 'мениск\\w*')
STEM_CRUCIATE = _rx('cruciate', 'cruzado', 'croise', 'kruisband', 'kreuzband', 'capraz bag\\w*', 'krizn\\w*', 'χιαστ\\w*', 'кръстн\\w*', '\\bacl\\b', '\\blca\\b', '\\bvkb\\b', '\\bocb\\b', '\\bacb\\b')
STEM_COLLATERAL = _rx('collateral\\w*', 'colateral\\w*', 'kollateral\\w*', 'collaterale\\w*', 'kolateraln\\w*', 'yan bag\\w*', 'πλαγι\\w*', 'колатерал\\w*', 'странич\\w*', 'innenband\\w*', 'binnenband\\w*', '\\bmcl\\b', '\\blcm\\b', '\\biyb\\b')
STEM_FRACTURE = _rx('fractur\\w*', 'fraktur\\w*', 'fractuur\\w*', '\\bfract\\b', 'kiri[kgğ]\\w*', 'prijelom\\w*', 'lom kosti', '\\bbreuk\\w*', '\\bbruch\\w*', 'καταγμα\\w*', 'καταγματ\\w*', 'фрактур\\w*', 'счупван\\w*', 'fisur\\w* (osea|oseas|kost)', 'fissur\\w* kost')
POSTERIOR_ONLY = _rx('\\bpcl\\b', '\\blcp\\b', '\\bhkb\\b', '\\bacb\\b', 'posterior cruciate', 'cruzado posterior', 'croise posterieur', 'achterste kruisband', 'hinteres kreuzband', 'arka capraz', 'straznji krizn', 'οπισθι[οα]\\w* χιαστ', 'задна кръстн', 'задната кръстн')
LATERAL_COLL_ONLY = _rx('\\blcl\\b', '\\bfcl\\b', 'lateral collateral', 'fibular collateral', 'colateral lateral', 'colateral externo', 'buitenband', 'aussenband', 'dis yan bag', 'lateralni kolateraln', 'εξω πλαγι', 'латерален колатерал')

def _near(clause: str, stem_rx: re.Pattern, qual_rx: re.Pattern, window: int=55):
    for m in stem_rx.finditer(clause):
        lo = max(0, m.start() - window)
        hi = min(len(clause), m.end() + window)
        if qual_rx.search(clause[lo:hi]):
            return True
    return False
STEM_RULES = {'ACL': (STEM_CRUCIATE, SIDE_ANTERIOR), 'MCL': (STEM_COLLATERAL, SIDE_MEDIAL), 'Medial Meniscus': (STEM_MENISCUS, SIDE_MEDIAL), 'Lateral Meniscus': (STEM_MENISCUS, SIDE_LATERAL)}

class _Matcher:

    def __init__(self, phrase_rx, stem=None, side=None, window=55):
        self.phrase_rx = phrase_rx
        self.stem = stem
        self.side = side
        self.window = window

    def search(self, clause):
        m = self.phrase_rx.search(clause)
        if m is not None:
            return m
        if self.stem is not None and _near(clause, self.stem, self.side, self.window):
            return self.stem.search(clause)
        return None
ANAT_MATCH = {t: _Matcher(ANAT[t], *STEM_RULES[t]) for t in PAIRED}
DIRECT_MATCH = {t: _Matcher(_rx(rx.pattern, STEM_FRACTURE.pattern) if t == 'Fracture' else rx) for t, rx in DIRECT.items()}
SEV_LOW = _rx('\\bsmall\\b', '\\bminimal\\b', '\\btrace\\b', '\\bmild\\b', '\\bslight\\b', '\\btiny\\b', '\\bscant\\b', '\\bdiscrete\\b', '\\blow ?grade\\b', '\\bincipient\\b', '\\bleve\\b', '\\bminim', '\\bpeque', '\\bfina\\b', '\\bfino\\b', '\\bligero\\b', '\\bescaso\\b', '\\bdiscreto\\b', '\\bhafif\\b', '\\baz miktarda\\b', '\\bsilik\\b', '\\bmanj\\w*', '\\bblago\\b', '\\bdiskretn', '\\bmalo\\b', '\\bpocetn', '\\bgering', '\\bdiskret', '\\bkleine?r?\\b', '\\bwenig\\b', '\\bzarte?\\b', '\\bbeperkte?\\b', '\\bgeringe\\b', '\\bweinig\\b', '\\blichte?\\b', '\\blicht\\b', '\\bηπι', '\\bμικρ', '\\bελαχιστ', '\\bαρχομεν', '\\bминимал', '\\bлек', '\\bмалк', '\\bнеголям')
SEV_HIGH = _rx('\\blarge\\b', '\\bmarked\\b', '\\bmassive\\b', '\\bsevere\\b', '\\bextensive\\b', '\\bmoderate\\b', '\\bgross\\b', '\\bsignificant\\b', '\\babundant\\b', '\\btense\\b', '\\bcomplete\\b', '\\bfull ?thickness\\b', '\\bhigh ?grade\\b', '\\badvanced\\b', '\\bmoderad', '\\bimportante\\b', '\\bsevera?\\b', '\\bmarcad', '\\bcuantios', '\\bespesor total\\b', '\\bcompleta?\\b', '\\bbelirgin\\b', '\\byaygin\\b', '\\bileri\\b', '\\bciddi\\b', '\\bbol\\b', '\\bkomplet', '\\bopsezan\\b', '\\bveliki\\b', '\\bizrazit', '\\bznacajn', '\\bumjeren', '\\buznapredoval', '\\bpotpun', '\\bkompleksn', '\\bausgepragt', '\\bdeutlich', '\\bmassiv', '\\bmassig', '\\bgross', '\\buitgebreid', '\\bgevorderd', '\\bveel\\b', '\\bmatige?\\b', '\\bvolledig', '\\bμετρι', '\\bμεγαλ', '\\bεκτεταμεν', '\\bευμεγεθ', '\\bσοβαρ', '\\bπληρη', '\\bголям', '\\bизразен', '\\bзначим', '\\bумерен', '\\bобилен', '\\bпълн')
GRADE_HIGH = re.compile('grade?[ao]?\\s*(3|4|iii|iv)\\b|icrs grade (iii|iv|3|4)|stupnja iv|stupnja iii|\\bgrado (3|4)\\b|\\bgrad (3|4)\\b|\\bgrade (3|4)\\b')
DEGENERATIVE_MARROW = _rx('subchondral', 'subcondral', 'subkondral', 'supkondraln', 'subchondraln', 'υποχονδρι', 'υπαρθρικ', 'субхондрал', 'subchondrale?', 'subartikuler', '\\bcyst', '\\bquist', '\\bzyste\\b', '\\bcistic', 'reactive', 'reactivo', 'degenerative', 'degenerativ', 'reaktiv', '\\bcisti\\b')
TRAUMA = _rx('\\bbruise\\b', '\\bcontusion', '\\bkontuz', '\\btrauma', '\\bimpaction\\b', '\\bpivot shift\\b', '\\bkissing\\b', '\\bacute\\b', '\\bagudo\\b', '\\bakut', '\\bpivot kaymasi\\b', '\\bcontusion osseuse\\b', '\\bbone bruise\\b', '\\bbotcontusie\\b', '\\bконтузион', '\\bμωλωπ', '\\bkontuzij', '\\bimpaktcij', '\\bimpakcij', '\\bfall\\b', '\\binjury\\b', '\\bimpression\\b')
SYNOVIAL_PROXY = _rx('bursit', 'burzit', '\\bbursa\\b[^.]{0,30}(fluid|distend|sivi|tekucin|opzetting)', 'suprapatellar (bursitis|effusion|recess)', 'suprapatellar bursa', 'suprapatellar bursada', 'suprapatelarno', 'suprapatellaire recessus', 'hoffa', 'hoffit', 'plica', 'plika', 'πλικα', 'fat pad[^.]{0,20}(edema|oedema)', 'kapsul', 'capsul', 'καψ', 'капсул', '\\bpannus\\b', '\\bsinov', '\\bsynov')

def _polarity(clause: str, span=None) -> str:
    if UNCERTAIN.search(clause):
        return 'uncertain'
    if span is None or not FEATURES['directional_negation']:
        if NEGATION.search(clause):
            return 'negative'
    elif _negated(clause, span[0], span[1]):
        return 'negative'
    if NORMALITY.search(clause):
        if TEAR.search(clause) or GRADE_HIGH.search(clause):
            return 'positive'
        return 'negative'
    return 'positive'

def _severity(clause: str) -> float:
    high = SEV_HIGH.search(clause) is not None
    low = SEV_LOW.search(clause) is not None
    if high and (not low):
        return 1.0
    if low and (not high):
        return 0.45
    if high and low:
        return 0.8
    return 0.75

def _grade(n_pos, n_neg, n_unc, best):
    if n_pos or n_unc:
        score = min(0.97, 0.5 + 0.45 * best + 0.015 * min(n_pos, 3))
        conf = min(1.0, 0.55 + 0.15 * n_pos)
    elif n_neg:
        score = max(0.04, 0.2 - 0.04 * n_neg)
        conf = min(0.9, 0.45 + 0.12 * n_neg)
    else:
        score, conf = (0.28, 0.05)
    return (score, conf)

def _paired_weight(clause: str, meniscus: bool) -> float:
    g = _grade_of(clause) if FEATURES['graded_pathology'] else None
    tear = TEAR.search(clause) is not None
    if meniscus:
        if tear:
            base = 1.0
        elif g is not None:
            base = 0.95 if g >= 3 else 0.3
        elif DEGEN.search(clause):
            base = 0.35
        else:
            base = 0.45
    elif tear:
        base = 1.0
    elif g is not None:
        base = 0.85 if g >= 2 else 0.3
    elif DEGEN.search(clause):
        base = 0.4
    else:
        base = 0.55
    if SEV_HIGH.search(clause) and (not SEV_LOW.search(clause)):
        base = min(1.0, base * 1.2)
    elif SEV_LOW.search(clause) and (not SEV_HIGH.search(clause)):
        base *= 0.7
    return base

def _score_paired(cls, tgt):
    anat_rx = ANAT_MATCH[tgt]
    path_rx = _rx(TEAR.pattern, DEGEN.pattern, INJURY.pattern)
    meniscus = 'Meniscus' in tgt
    n_pos = n_neg = n_unc = 0
    best = 0.0
    for c in cls:
        hit = anat_rx.search(c)
        if hit is None and meniscus and PLURAL_MENISCI.search(c) and (not ANY_SIDE.search(c)):
            hit = PLURAL_MENISCI.search(c)
        if hit is None:
            continue
        pm = path_rx.search(c)
        if pm is None and _grade_of(c) is None:
            if NORMAL_PHRASE.search(c) or (NORMALITY.search(c) and (not NEGATION.search(c))):
                n_neg += 1
            continue
        span = (pm.start(), pm.end()) if pm is not None else None
        pol = _polarity(c, span)
        if pol == 'positive':
            n_pos += 1
            best = max(best, _paired_weight(c, meniscus))
        elif pol == 'negative':
            n_neg += 1
        else:
            n_unc += 1
            best = max(best, 0.45 * _paired_weight(c, meniscus))
    s, cf = _grade(n_pos, n_neg, n_unc, best)
    return (s, cf, n_pos, n_neg)

def _score_clauses(cls, anat_rx, path_rx=None, decoy_rx=None, context_penalty=None, context_bonus=None):
    n_pos = n_neg = n_unc = 0
    best = 0.0
    for c in cls:
        m = anat_rx.search(c)
        if not m:
            continue
        if decoy_rx is not None and decoy_rx.search(c):
            continue
        if path_rx is not None and (not path_rx.search(c)):
            if NORMAL_PHRASE.search(c) or (NORMALITY.search(c) and (not NEGATION.search(c))):
                n_neg += 1
            continue
        pol = _polarity(c, (m.start(), m.end()))
        if pol == 'positive':
            n_pos += 1
            w = _severity(c)
            if context_penalty is not None and context_penalty.search(c):
                w *= 0.45
            if context_bonus is not None and context_bonus.search(c):
                w = min(1.0, w * 1.35)
            best = max(best, w)
        elif pol == 'negative':
            n_neg += 1
        else:
            n_unc += 1
            best = max(best, 0.3)
    s, c = _grade(n_pos, n_neg, n_unc, best)
    return (s, c, n_pos, n_neg)

def _score_oa(cls):
    acc = {t: {'pos': 0, 'neg': 0, 'unc': 0, 'best': 0.0} for t in OA_TARGETS}
    g_pos, g_neg, g_best = (0, 0, 0.0)
    for c in cls:
        m = OA_EVIDENCE.search(c)
        if not m:
            continue
        pol = _polarity(c, (m.start(), m.end()))
        sev = _severity(c)
        tf_med = _near(c, TF_SITE, SIDE_MEDIAL, 45)
        tf_lat = _near(c, TF_SITE, SIDE_LATERAL, 45)
        pf = PF_SITE.search(c) is not None
        hits = []
        if tf_med:
            hits.append('Medial OA')
        if tf_lat:
            hits.append('Lateral OA')
        if pf:
            hits.append('PF OA')
        if not hits:
            if pol == 'positive':
                g_pos += 1
                g_best = max(g_best, sev if GLOBAL_OA.search(c) else sev * 0.7)
            elif pol == 'negative':
                g_neg += 1
            continue
        for t in hits:
            if pol == 'positive':
                acc[t]['pos'] += 1
                acc[t]['best'] = max(acc[t]['best'], sev)
            elif pol == 'negative':
                acc[t]['neg'] += 1
            else:
                acc[t]['unc'] += 1
                acc[t]['best'] = max(acc[t]['best'], 0.3)
    out = {}
    for t in OA_TARGETS:
        a = acc[t]
        pos, neg, unc, best = (a['pos'], a['neg'], a['unc'], a['best'])
        if not (pos or unc) and g_pos and FEATURES['oa_inherit']:
            if neg:
                score, conf = _grade(0, neg, 0, 0.0)
                score = max(score, 0.35)
                conf *= 0.7
            else:
                score, conf = _grade(g_pos, 0, 0, g_best * 0.92)
                conf *= 0.75
        else:
            score, conf = _grade(pos, neg + g_neg, unc, best)
        out[t] = (score, conf, pos, neg)
    return out

def extract(report: str) -> dict:
    cls = clauses(report)
    out = {}
    for tgt in PAIRED:
        s, c, npos, nneg = _score_paired(cls, tgt)
        out[tgt] = s
        out[tgt + '__conf'] = c
        out[tgt + '__npos'] = npos
        out[tgt + '__nneg'] = nneg
    for tgt, (s, c, npos, nneg) in _score_oa(cls).items():
        out[tgt] = s
        out[tgt + '__conf'] = c
        out[tgt + '__npos'] = npos
        out[tgt + '__nneg'] = nneg
    for tgt in ('Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture'):
        if tgt == 'Contusion':
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt), context_penalty=DEGENERATIVE_MARROW, context_bonus=TRAUMA)
        else:
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt))
        out[tgt] = s
        out[tgt + '__conf'] = c
        out[tgt + '__npos'] = npos
        out[tgt + '__nneg'] = nneg
    if FEATURES['synovitis_backoff'] and out['Synovitis__npos'] == 0 and (out['Synovitis__nneg'] == 0):
        proxy = sum((1 for c in cls if SYNOVIAL_PROXY.search(c) and _polarity(c) == 'positive'))
        eff = out['Effusion']
        prior = 0.3 + 0.3 * max(0.0, (eff - 0.5) / 0.45) + 0.06 * min(proxy, 3)
        out['Synovitis'] = min(0.72, prior)
        out['Synovitis__conf'] = 0.18
    return out

FEATURES['synovitis_backoff'] = False


## Configuration and reproducibility

In [ ]:
_IMPLEMENTATION_SHA = "74eb678d33a67439899990276f44ad34651a00e78065c841546d1a127c4ad08d"
import contextlib
import dataclasses
import gc
import hashlib
import json
import math
import os
import random
import shutil
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from PIL import Image
from sklearn.metrics import roc_auc_score
import timm
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset


UID = "StudyInstanceUID"
SLOTS = [(plane, fs) for plane in ("Sagittal", "Coronal", "Axial") for fs in (True, False)]
VERSION = "rsna-rules-mil-v1"


@dataclasses.dataclass
class Config:
    mode: str = "train"  # train, infer, or train_infer; infer never trains or downloads weights
    data_root: str = ""  # auto-discover the competition mount
    output_dir: str = "/kaggle/working/rsna_artifacts"
    checkpoint_dir: str = ""  # attached training output, required in infer mode
    image_size: int = 224  # divisible by 14 and 16: both DINO families
    slices: int = 8
    crop_mm: float = 140.0
    slice_band: tuple = (0.10, 0.90)
    seed: int = 42
    folds: int = 5
    train_folds: tuple = (0, 1, 2, 3, 4)
    head_epochs: int = 10
    patience: int = 3
    head_batch: int = 64
    head_lr: float = 3e-4
    finetune_epochs: int = 0  # frozen baseline first; set 2 for fixed, more expensive fine-tuning
    unfreeze_blocks: int = 2
    backbone_lr: float = 1e-5
    image_batch: int = 1
    accumulation: int = 8
    encode_batch: int = 24
    hidden: int = 256
    dropout: float = 0.20
    weak_weight: float = 0.50
    gold_weight: float = 3.0
    confidence_min: float = 0.15
    workers: int = 2
    header_workers: int = 4
    require_gpu: bool = True
    allow_random_init: bool = False  # synthetic engineering checks only
    resume: bool = True


@dataclasses.dataclass
class Member:
    name: str = "dinov2"
    backbone: str = "vit_small_patch14_dinov2.lvd142m"
    pretrained_path: str = ""  # timm backbone state_dict; classifier removed
    pretrained_sha256: str = ""  # mandatory for an attached file
    provenance: str = "DINOv2 self-supervised image pretraining; no competition labels"
    normalization: str = "imagenet"  # imagenet or minus_one_one, match source weights
    weight: float = 1.0
    targets: tuple = tuple(TARGETS)  # specialist contributes only to these findings


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()


def file_sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, indent=2, default=str, allow_nan=False))
    os.replace(temp, path)


def preprocessing(cfg):
    return {k: getattr(cfg, k) for k in ("image_size", "slices", "crop_mm", "slice_band")}


def find_data_root(cfg):
    candidates = [Path(cfg.data_root)] if cfg.data_root else [
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"), Path("data")]
    if not cfg.data_root and Path("/kaggle/input").exists():
        candidates += list(Path("/kaggle/input").glob("*"))
        candidates += list(Path("/kaggle/input").glob("*/*"))
    for path in candidates:
        if (path / "test.csv").is_file() and (path / "test_series.csv").is_file():
            return path.resolve()
    raise FileNotFoundError("Attach RSNA Knee Abnormality Detection competition data or set data_root.")


def read_table(root, name):
    frame = pd.read_csv(root / name, dtype={UID: str, "SeriesInstanceUID": str})
    if UID not in frame or frame[UID].isna().any():
        raise ValueError(f"{name}: missing study identifiers")
    return frame


def validate_studies(frame, tag):
    if frame[UID].duplicated().any() or len(frame) == 0:
        raise ValueError(f"{tag}: duplicate identifiers or empty table")




## Experiment settings

Edit this cell before training. Inference restores image preprocessing and head architecture from your checkpoints.

In [ ]:
CFG = Config(
    mode=RUN_MODE,
    checkpoint_dir=CHECKPOINT_DIR,
    # data_root="/kaggle/input/competitions/rsna-knee-abnormality-detection",
    output_dir="/kaggle/working/rsna_artifacts",
    image_size=224,
    slices=8,
    folds=5,
    train_folds=(0, 1, 2, 3, 4),
    head_epochs=10,
    finetune_epochs=0,  # set 2 for optional late-block fine-tuning; use a new output_dir
    image_batch=1,
    encode_batch=24,  # reduce to 8 or 12 if your GPU runs out of memory
    workers=2,
)
MEMBERS = [Member()]

# Optional DINOv3 member. Append BEFORE training, in a new experiment directory.
# Its public foundation may require accepting the upstream weight license/access terms.
# Attach a compatible timm state_dict with an independently obtained SHA-256 when offline.
# MEMBERS.append(Member(
#     name="dinov3", backbone="vit_small_patch16_dinov3.lvd1689m",
#     pretrained_path="/kaggle/input/your-foundation/dinov3_timm.pt",
#     pretrained_sha256="YOUR_VERIFIED_SHA256",
#     provenance="Public DINOv3 self-supervised foundation weights; no knee labels"))

# Optional RadImageNet member: official Keras weights cannot be loaded into timm!
# Supply a public PyTorch ResNet-50 state_dict matching timm with classifier removed.
# Confirm the source's preprocessing convention (this example uses [-1,1]).
# MEMBERS.append(Member(
#     name="radimagenet", backbone="resnet50",
#     pretrained_path="/kaggle/input/your-foundation/radimagenet_timm.pt",
#     pretrained_sha256="YOUR_VERIFIED_SHA256", normalization="minus_one_one",
#     provenance="Public RadImageNet foundation; compatible conversion, no knee labels"))

# Optional specialist: a separate head/fold model, contributing only to its named target.
# This costs another model training run; add only when independent validation supports it.
# MEMBERS.append(Member(name="synovitis_specialist", targets=("Synovitis",), weight=0.25))

# For long training runs, select one or more folds, then continue into the SAME output_dir
# with the SAME members/config. Completed folds resume; partially trained ensembles cannot
# produce submissions. Snapshot/copy prior checkpoints before a new Kaggle session.
# CFG.train_folds = (0,)

print("Selected members:", [(m.name, m.backbone, m.targets) for m in MEMBERS])


## Deterministic labels and leakage-aware folds

In [ ]:
def make_targets(train, cfg):
    """Only train.csv reports and official binary annotations can supply labels."""
    if "Report" not in train or any(t not in train for t in TARGETS):
        raise ValueError("train.csv must contain Report and all twelve official target columns")
    rules = pd.DataFrame([extract(r) for r in train.Report.fillna("")])
    y = rules[TARGETS].to_numpy(np.float32)
    confidence = rules[[t + "__conf" for t in TARGETS]].to_numpy(np.float32)
    mentioned = np.column_stack([
        rules[t + "__npos"].to_numpy() + rules[t + "__nneg"].to_numpy() > 0 for t in TARGETS])
    weights = cfg.weak_weight * confidence
    weights[(confidence < cfg.confidence_min) | ~mentioned] = 0
    gold = train[TARGETS].notna().to_numpy()
    exact = train[TARGETS].to_numpy(np.float32)
    if not np.isin(exact[gold], [0.0, 1.0]).all():
        raise ValueError("Official expert labels must be 0/1 or missing")
    y[gold], weights[gold] = exact[gold], cfg.gold_weight
    if not np.isfinite(y).all() or weights.sum() <= 0:
        raise ValueError("No usable supervision or nonfinite rule outputs")
    audit = pd.DataFrame({"target": TARGETS, "gold_count": gold.sum(0),
                          "weak_observed": ((weights > 0) & ~gold).sum(0),
                          "masked_unknown": (weights == 0).sum(0)})
    return y, weights.astype(np.float32), gold, audit


def report_patient_groups(train):
    """Union exact duplicate reports AND available patient IDs; blank reports stay separate."""
    n = len(train)
    parent = np.arange(n)

    def root(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def unite_keys(keys):
        seen = {}
        for i, key in enumerate(keys):
            if key is None:
                continue
            if key in seen:
                parent[root(i)] = root(seen[key])
            else:
                seen[key] = i

    texts = [" ".join(normalize(r).split()) for r in train.Report.fillna("")]
    unite_keys([digest(r) if r else None for r in texts])
    for column in ("PatientID", "PatientInstanceUID", "patient_id"):
        if column in train:
            unite_keys([str(v) if pd.notna(v) and str(v).strip() else None for v in train[column]])
    return np.array([str(root(i)) for i in range(n)])


def balanced_group_folds(y, gold, groups, n_folds, seed):
    """Greedy grouped multilabel balancing; rare gold positives and negatives get priority."""
    unique, inverse = np.unique(groups, return_inverse=True)
    if len(unique) < n_folds:
        raise ValueError("Too few independent groups for the requested folds")
    flags = np.concatenate([gold * (y >= .5), gold * (y < .5),
                            (y >= .5).astype(float) * .05, np.ones((len(y), 1))], axis=1)
    sums = np.zeros((len(unique), flags.shape[1]))
    np.add.at(sums, inverse, flags)
    desired = np.maximum(sums.sum(0) / n_folds, 1.0)
    rng = np.random.default_rng(seed)
    priority = (sums / desired).max(1) + rng.uniform(0, 1e-6, len(unique))
    order = np.argsort(-priority)
    totals = np.zeros((n_folds, flags.shape[1]))
    counts = np.zeros(n_folds)
    assignment = np.zeros(len(unique), int)
    for position, g in enumerate(order):
        scores = []
        for f in range(n_folds):
            candidate = totals.copy()
            candidate[f] += sums[g]
            scores.append(np.square((candidate - candidate.mean(0)) / desired).sum())
        eligible = np.flatnonzero(counts == 0) if position < n_folds else np.arange(n_folds)
        f = int(eligible[np.argmin(np.array(scores)[eligible])])
        assignment[g] = f
        totals[f] += sums[g]
        counts[f] += 1
    return assignment[inverse]


def auc_report(y, predictions, mask):
    rows = []
    for j, target in enumerate(TARGETS):
        valid = mask[:, j] & np.isfinite(predictions[:, j])
        labels = y[valid, j]
        auc = float(roc_auc_score(labels, predictions[valid, j])) if len(np.unique(labels)) == 2 else None
        rows.append({"target": target, "auc": auc, "n": int(valid.sum()),
                     "positives": int((labels == 1).sum()), "negatives": int((labels == 0).sum())})
    values = [r["auc"] for r in rows if r["auc"] is not None]
    return {"macro_over_defined_targets": float(np.mean(values)) if values else None,
            "defined_targets": len(values), "total_targets": len(TARGETS), "per_target": rows}




## DICOM geometry, sequence slots, and bounded image caches

In [ ]:
def plane_from_header(ds):
    try:
        orientation = np.asarray(ds.ImageOrientationPatient, float)
        normal = np.cross(orientation[:3], orientation[3:])
        return ["Sagittal", "Coronal", "Axial"][int(np.argmax(np.abs(normal)))]
    except (AttributeError, ValueError, TypeError):
        return None


def infer_fatsat(ds):
    text = " ".join(str(getattr(ds, k, "")) for k in ("SeriesDescription", "SequenceName", "ScanOptions"))
    return bool(__import__("re").search(r"(?i)(?:\bfs\b|fat.?sat|fat.?sup|stir|spair|spir|dixon.*water)", text))


def flag(value, fallback):
    if pd.isna(value):
        return fallback
    if str(value).strip().lower() in ("1", "1.0", "true", "yes"):
        return True
    if str(value).strip().lower() in ("0", "0.0", "false", "no"):
        return False
    raise ValueError(f"Unrecognized series flag: {value!r}")


def locate_series(root, split, study, series):
    for path in (root / f"{split}_series" / study / series, root / f"{split}_series" / series):
        if path.is_dir():
            return path
    raise FileNotFoundError(f"Missing {split} MRI series for study={study}, series={series}")


def describe_series(row, root, split):
    study, series = str(row[UID]), str(row["SeriesInstanceUID"])
    directory = locate_series(root, split, study, series)
    files = sorted(directory.glob("*.dcm"))
    if not files:
        raise ValueError(f"Empty DICOM series {directory}")
    ds = pydicom.dcmread(files[len(files) // 2], stop_before_pixels=True)
    supplied = str(row.get("Anatomical_Plane", "")).strip().capitalize()
    plane = supplied if supplied in {s[0] for s in SLOTS} else plane_from_header(ds)
    if plane is None:
        raise ValueError(f"Cannot determine anatomical plane: {series}")
    fs = flag(row.get("Fat_Suppression", np.nan), infer_fatsat(ds))
    text = str(getattr(ds, "SeriesDescription", "")).lower()
    fluid_guess = not ("t1" in text and "t2" not in text and "pd" not in text)
    fluid = flag(row.get("Fluid_Sensitive", np.nan), fluid_guess)
    side = str(getattr(ds, "ImageLaterality", getattr(ds, "Laterality", ""))).upper()
    if side not in ("L", "R"):
        try:
            iop = np.asarray(ds.ImageOrientationPatient, float)
            ipp = np.asarray(ds.ImagePositionPatient, float)
            ps = np.asarray(ds.PixelSpacing, float)
            center = ipp + iop[:3] * ps[1] * (ds.Columns - 1) / 2 + iop[3:] * ps[0] * (ds.Rows - 1) / 2
            side = "L" if center[0] > 15 else "R" if center[0] < -15 else "unknown"
        except (AttributeError, ValueError, TypeError):
            side = "unknown"
    return {"study": study, "series": series, "plane": plane, "fs": fs, "fluid": fluid,
            "side": side, "files": [str(p) for p in files],
            "orientation_missing": not hasattr(ds, "ImageOrientationPatient"),
            "plane_disagreement": plane_from_header(ds) not in (None, plane)}


def ordered_files(record):
    axis = {"Sagittal": 0, "Coronal": 1, "Axial": 2}[record["plane"]]
    values = []
    for path in record["files"]:
        ds = pydicom.dcmread(path, stop_before_pixels=True,
                            specific_tags=["ImagePositionPatient", "InstanceNumber"])
        position = getattr(ds, "ImagePositionPatient", None)
        instance = getattr(ds, "InstanceNumber", None)
        values.append((path, float(position[axis]) if position is not None else None,
                       int(instance) if instance is not None else None))
    if all(v[1] is not None for v in values) and len({v[1] for v in values}) > 1:
        values.sort(key=lambda v: (v[1], v[2] if v[2] is not None else 0, v[0]))
        record["order_method"] = "patient_geometry"
    elif all(v[2] is not None for v in values):
        values.sort(key=lambda v: (v[2], v[0]))
        record["order_method"] = "instance_number"
    else:
        raise ValueError(f"No reliable geometric or instance order for series {record['series']}")
    paths = [v[0] for v in values]
    if record["plane"] == "Sagittal" and record["side"] == "R":
        paths.reverse()
    return paths


def canonical_orientation(array, ds, plane, return_transpose=False):
    """Signed transpose/flips into patient axes; retains obliquity rather than resampling it."""
    transposed = False
    try:
        iop = np.asarray(ds.ImageOrientationPatient, float)
        col, row = iop[:3], iop[3:]
        horizontal, vertical = {"Sagittal": (1, 2), "Coronal": (0, 2), "Axial": (0, 1)}[plane]
        down_sign = 1 if plane == "Axial" else -1
        if abs(row[horizontal]) > abs(col[horizontal]):
            array = array.T
            col, row = row, col
            transposed = True
        if col[horizontal] < 0:
            array = array[:, ::-1]
        if row[vertical] * down_sign < 0:
            array = array[::-1]
    except (AttributeError, TypeError, ValueError):
        pass  # orientation absence is recorded by the header/ordering audit
    return (array, transposed) if return_transpose else array


def render_slice(path, plane, side, cfg):
    ds = pydicom.dcmread(path)
    pixels = ds.pixel_array
    if pixels.ndim != 2:
        raise ValueError("Expected a single-frame monochrome DICOM")
    array = pixels.astype(np.float32)
    padding = getattr(ds, "PixelPaddingValue", None)
    valid = np.isfinite(array) if padding is None else np.isfinite(array) & (pixels != float(padding))
    array = array * float(getattr(ds, "RescaleSlope", 1)) + float(getattr(ds, "RescaleIntercept", 0))
    if not hasattr(ds, "PixelSpacing"):
        raise ValueError("Missing PixelSpacing; physical crop cannot be established")
    spacing = np.asarray(ds.PixelSpacing, float)
    if len(spacing) != 2 or not np.isfinite(spacing).all() or (spacing <= 0).any():
        raise ValueError("Invalid PixelSpacing")
    height = min(array.shape[0], max(2, round(cfg.crop_mm / spacing[0])))
    width = min(array.shape[1], max(2, round(cfg.crop_mm / spacing[1])))
    top, left = (array.shape[0] - height) // 2, (array.shape[1] - width) // 2
    array, valid = array[top:top + height, left:left + width], valid[top:top + height, left:left + width]
    values = array[valid]
    if len(values) < 4:
        raise ValueError("Insufficient valid pixels")
    low, high = np.percentile(values, [1, 99])
    if high <= low:
        raise ValueError("Constant/blank MRI slice")
    array = np.clip((array - low) / (high - low), 0, 1)
    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        array = 1 - array
    array[~valid] = 0
    array, transposed = canonical_orientation(array, ds, plane, return_transpose=True)
    physical_height, physical_width = height * spacing[0], width * spacing[1]
    if transposed:
        physical_height, physical_width = physical_width, physical_height
    if side == "R" and plane in ("Coronal", "Axial"):
        array = array[:, ::-1]
    # Letterbox preserves physical aspect ratio after crop and canonical transpose.
    image = Image.fromarray(np.uint8(np.rint(array * 255)))
    ratio = cfg.image_size / max(physical_width, physical_height)
    size = (max(1, round(physical_width * ratio)), max(1, round(physical_height * ratio)))
    image = image.resize(size, Image.Resampling.BILINEAR)
    canvas = Image.new("L", (cfg.image_size, cfg.image_size))
    canvas.paste(image, ((cfg.image_size - size[0]) // 2, (cfg.image_size - size[1]) // 2))
    return np.asarray(canvas)


def build_image_cache(root, split, studies, cfg):
    series = read_table(root, f"{split}_series.csv")
    series = series[series[UID].isin(studies[UID])].copy()
    if series[[UID, "SeriesInstanceUID"]].duplicated().any():
        raise ValueError("Duplicate MRI series descriptors")
    key = digest({"version": VERSION, "implementation": implementation_hash(), "root": str(root), "split": split,
                  "ids": studies[UID].tolist(), "series_csv": file_sha(root / f"{split}_series.csv"),
                  "preprocess": preprocessing(cfg)})[:20]
    folder = Path(cfg.output_dir) / "images" / f"{split}_{key}"
    folder.mkdir(parents=True, exist_ok=True)
    receipt_path = folder / "receipt.json"
    if receipt_path.exists():
        receipt = json.loads(receipt_path.read_text())
        paths = [folder / (digest(u)[:24] + ".npz") for u in studies[UID]]
        if all(p.exists() for p in paths) and receipt.get("complete"):
            print(f"Reuse {split} image cache: {len(paths)} studies")
            return folder, receipt
    estimated = len(studies) * len(SLOTS) * cfg.slices * cfg.image_size ** 2
    if shutil.disk_usage(folder).free < estimated * 1.05:
        raise RuntimeError(f"Need up to {estimated / 2**30:.1f} GiB free for {split} image cache")
    with ThreadPoolExecutor(max_workers=cfg.header_workers) as pool:
        records = list(pool.map(lambda row: describe_series(row, root, split), series.to_dict("records")))
    grouped = {}
    for rec in records:
        grouped.setdefault(rec["study"], []).append(rec)
    receipt = {"complete": False, "ids": studies[UID].tolist(), "cache_key": key,
               "preprocessing": preprocessing(cfg), "selected": {}, "decode_errors": [],
               "unknown_side_series": sum(r["side"] == "unknown" for r in records),
               "orientation_missing_series": sum(r["orientation_missing"] for r in records),
               "plane_disagreements": sum(r["plane_disagreement"] for r in records)}
    for index, study in enumerate(studies[UID]):
        path = folder / (digest(study)[:24] + ".npz")
        images = np.zeros((len(SLOTS), cfg.slices, cfg.image_size, cfg.image_size), np.uint8)
        mask = np.zeros((len(SLOTS), cfg.slices), bool)
        selected = []
        for slot, (plane, fs) in enumerate(SLOTS):
            candidates = [r for r in grouped.get(study, []) if r["plane"] == plane and r["fs"] == fs]
            if not candidates:
                continue
            rec = sorted(candidates, key=lambda r: (r["fluid"] == fs, len(r["files"]), r["series"]), reverse=True)[0]
            files = ordered_files(rec)
            positions = np.rint(np.linspace(cfg.slice_band[0] * (len(files) - 1),
                                             cfg.slice_band[1] * (len(files) - 1), cfg.slices)).astype(int)
            selected.append({"slot": slot, "series": rec["series"], "side": rec["side"],
                             "slice_order": rec["order_method"]})
            for k, position in enumerate(positions):
                try:
                    images[slot, k] = render_slice(files[position], plane, rec["side"], cfg)
                    mask[slot, k] = True
                except Exception as exc:
                    receipt["decode_errors"].append({"study": study, "series": rec["series"],
                                                     "file": Path(files[position]).name,
                                                     "error": str(exc)[:240]})
        if not mask.any():
            atomic_json(folder / "failed_audit.json", receipt)
            raise RuntimeError(f"Study {study} has no usable images; inspect failed_audit.json")
        temp = path.with_suffix(".tmp.npz")
        np.savez_compressed(temp, images=images, mask=mask)
        os.replace(temp, path)
        receipt["selected"][study] = selected
        if (index + 1) % 100 == 0 or index + 1 == len(studies):
            print(f"{split} image cache {index + 1}/{len(studies)}", flush=True)
    receipt["complete"] = True
    atomic_json(receipt_path, receipt)
    return folder, receipt


class ImageDataset(Dataset):
    def __init__(self, folder, ids, y=None, weights=None):
        self.folder, self.ids, self.y, self.weights = Path(folder), list(ids), y, weights

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        with np.load(self.folder / (digest(self.ids[i])[:24] + ".npz"), allow_pickle=False) as data:
            images, mask = torch.from_numpy(data["images"].copy()), torch.from_numpy(data["mask"].copy())
        item = {"x": images, "mask": mask}
        if self.y is not None:
            item.update(y=torch.from_numpy(self.y[i]), w=torch.from_numpy(self.weights[i]))
        return item




## Frozen image encoders, cross-series label attention, and feature caches

In [ ]:
def amp_context(device):
    if device.type != "cuda":
        return contextlib.nullcontext()
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.autocast("cuda", dtype=dtype)


class FindingHead(nn.Module):
    def __init__(self, dim, cfg):
        super().__init__()
        h = cfg.hidden
        if h % 4:
            raise ValueError("hidden must be divisible by four attention heads")
        self.projection = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, h), nn.GELU())
        self.slot = nn.Parameter(torch.randn(len(SLOTS), h) * .02)
        self.position = nn.Parameter(torch.randn(cfg.slices, h) * .02)
        self.query = nn.Parameter(torch.randn(len(TARGETS), h) * .02)
        self.attention = nn.MultiheadAttention(h, 4, batch_first=True, dropout=cfg.dropout)
        self.out = nn.Sequential(nn.LayerNorm(3 * h), nn.Dropout(cfg.dropout),
                                 nn.Linear(3 * h, h), nn.GELU(), nn.Linear(h, 1))

    def forward(self, features, mask):
        if not mask.flatten(1).any(1).all():
            raise ValueError("At least one valid MRI token is required per study")
        x = self.projection(features)
        x = x + self.slot[None, :, None, :] + self.position[None, None, :, :]
        x, valid = x.flatten(1, 2), mask.flatten(1)
        queries = self.query[None].expand(len(x), -1, -1)
        attended, _ = self.attention(queries, x, x, key_padding_mask=~valid, need_weights=False)
        mean = (x * valid[..., None]).sum(1) / valid.sum(1, keepdim=True).clamp_min(1)
        maximum = x.masked_fill(~valid[..., None], -torch.inf).amax(1)
        pooled = torch.cat([attended, mean[:, None].expand_as(attended),
                            maximum[:, None].expand_as(attended)], dim=-1)
        return self.out(pooled).squeeze(-1)


class StudyModel(nn.Module):
    def __init__(self, member, cfg, pretrained=False):
        super().__init__()
        if member.normalization not in ("imagenet", "minus_one_one"):
            raise ValueError("Unknown normalization recipe")
        options = {"num_classes": 0}
        if "vit_" in member.backbone:
            options["img_size"] = cfg.image_size
        self.backbone = timm.create_model(member.backbone, pretrained=pretrained, **options)
        self.head = FindingHead(self.backbone.num_features, cfg)
        self.encode_batch = cfg.encode_batch
        mean, std = ([.485, .456, .406], [.229, .224, .225]) if member.normalization == "imagenet" else ([.5] * 3, [.5] * 3)
        self.register_buffer("mean", torch.tensor(mean).reshape(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).reshape(1, 3, 1, 1))
        for parameter in self.backbone.parameters():
            parameter.requires_grad_(False)

    def train(self, mode=True):
        super().train(mode)
        # Keep foundation BatchNorm statistics fixed, including during fine-tuning.
        self.backbone.eval()
        return self

    def encode(self, images, mask):
        shape = images.shape
        flat_mask = mask.flatten()
        indices = flat_mask.nonzero(as_tuple=True)[0]
        x = images.flatten(0, 2)[indices].float().unsqueeze(1).expand(-1, 3, -1, -1) / 255.0
        x = (x - self.mean) / self.std
        encoded = torch.cat([self.backbone(part) for part in x.split(self.encode_batch)], dim=0)
        features = encoded.new_zeros((flat_mask.numel(), encoded.shape[-1])).index_copy(0, indices, encoded)
        return features.reshape(shape[0], shape[1], shape[2], -1)

    def forward(self, images, mask):
        return self.head(self.encode(images, mask), mask)

    def unfreeze(self, n_blocks):
        if n_blocks <= 0:
            raise ValueError("Fine-tuning needs unfreeze_blocks > 0")
        if hasattr(self.backbone, "blocks"):
            parts = list(self.backbone.blocks)[-n_blocks:]
            if hasattr(self.backbone, "norm"):
                parts.append(self.backbone.norm)
            if hasattr(self.backbone, "set_grad_checkpointing"):
                self.backbone.set_grad_checkpointing(True)
        elif hasattr(self.backbone, "layer4"):
            parts = [self.backbone.layer4]
        else:
            raise ValueError("Fine-tuning recipe supports timm ViT and ResNet encoders")
        for part in parts:
            for parameter in part.parameters():
                parameter.requires_grad_(True)


def load_foundation(member, cfg, device):
    if member.pretrained_path:
        path = Path(member.pretrained_path)
        if not member.pretrained_sha256 or file_sha(path) != member.pretrained_sha256:
            raise ValueError(f"{member.name}: supply the verified SHA-256 of the foundation weight file")
        model = StudyModel(member, cfg, pretrained=False)
        state = torch.load(path, map_location="cpu", weights_only=True)
        state = state.get("state_dict", state)
        model.backbone.load_state_dict(state, strict=True)
    else:
        if "radimagenet" in member.name.lower() or "radimagenet" in member.provenance.lower():
            raise ValueError("RadImageNet requires a compatible public foundation file and its checksum")
        model = StudyModel(member, cfg, pretrained=not cfg.allow_random_init)
    return model.to(device).eval()


def backbone_sha(model):
    h = hashlib.sha256()
    for key, tensor in sorted(model.backbone.state_dict().items()):
        h.update(key.encode())
        h.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


def data_loader(dataset, cfg, batch_size, shuffle=False, seed=None):
    generator = torch.Generator().manual_seed(cfg.seed if seed is None else seed)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=cfg.workers,
                      pin_memory=torch.cuda.is_available(), generator=generator,
                      persistent_workers=cfg.workers > 0)


def extract_feature_cache(model, folder, ids, receipt, cfg, device):
    weight_hash = backbone_sha(model)
    key = digest({"image_key": receipt["cache_key"], "backbone": weight_hash,
                  "normalization": [model.mean.flatten().tolist(), model.std.flatten().tolist()]})[:20]
    base = Path(cfg.output_dir) / "features"
    base.mkdir(parents=True, exist_ok=True)
    x_path, m_path, ready = base / f"{key}.npy", base / f"{key}_mask.npy", base / f"{key}.json"
    if ready.exists() and x_path.exists() and m_path.exists():
        metadata = json.loads(ready.read_text())
        if metadata["ids"] == list(ids) and metadata["backbone_sha256"] == weight_hash:
            return np.load(x_path, mmap_mode="r"), np.load(m_path, mmap_mode="r")
    dim = model.backbone.num_features
    shape = (len(ids), len(SLOTS), cfg.slices, dim)
    if shutil.disk_usage(base).free < np.prod(shape) * 2 * 1.05:
        raise RuntimeError("Insufficient disk space for encoder feature cache")
    temp_x, temp_m = x_path.with_suffix(".tmp.npy"), m_path.with_suffix(".tmp.npy")
    features = np.lib.format.open_memmap(temp_x, mode="w+", dtype=np.float16, shape=shape)
    masks = np.lib.format.open_memmap(temp_m, mode="w+", dtype=bool, shape=shape[:3])
    loader = data_loader(ImageDataset(folder, ids), cfg, cfg.image_batch)
    offset = 0
    model.eval()
    with torch.inference_mode():
        for batch in loader:
            images, mask = batch["x"].to(device), batch["mask"].to(device)
            with amp_context(device):
                values = model.encode(images, mask)
            n = len(images)
            values = values.float().cpu().numpy()
            if not np.isfinite(values).all() or (np.abs(values) > 65504).any():
                raise ValueError("Nonfinite/out-of-range encoder features")
            features[offset:offset + n] = values.astype(np.float16)
            masks[offset:offset + n] = mask.cpu().numpy()
            offset += n
            if offset % 100 == 0 or offset == len(ids):
                print(f"Encoder features {offset}/{len(ids)}", flush=True)
    features.flush()
    masks.flush()
    del features, masks
    os.replace(temp_x, x_path)
    os.replace(temp_m, m_path)
    atomic_json(ready, {"ids": list(ids), "backbone_sha256": weight_hash, "shape": shape})
    return np.load(x_path, mmap_mode="r"), np.load(m_path, mmap_mode="r")


class FeatureDataset(Dataset):
    def __init__(self, features, masks, indices, y=None, weights=None):
        self.features, self.masks, self.indices = features, masks, np.asarray(indices)
        self.y, self.weights = y, weights

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        i = self.indices[index]
        item = {"x": torch.from_numpy(np.array(self.features[i], dtype=np.float32)),
                "mask": torch.from_numpy(np.array(self.masks[i]))}
        if self.y is not None:
            item.update(y=torch.from_numpy(self.y[i]), w=torch.from_numpy(self.weights[i]))
        return item




## Training, independent inner epoch selection, and full-fold refit

In [ ]:
def weighted_bce(logits, y, weights):
    if not torch.isfinite(logits).all():
        raise ValueError("Nonfinite model logits")
    return (F.binary_cross_entropy_with_logits(logits.float(), y.float(), reduction="none") * weights).sum() / weights.sum().clamp_min(1e-8)


def evaluate_loss(head, loader, device):
    numerator = denominator = 0.0
    head.eval()
    with torch.inference_mode():
        for batch in loader:
            x, mask, y, w = (batch[k].to(device) for k in ("x", "mask", "y", "w"))
            with amp_context(device):
                logits = head(x, mask)
            numerator += float((F.binary_cross_entropy_with_logits(logits.float(), y, reduction="none") * w).sum())
            denominator += float(w.sum())
    if denominator <= 0:
        raise ValueError("Inner validation has no observed labels for this member")
    return numerator / denominator


def head_epoch(head, loader, optimizer, device):
    head.train()
    total = count = 0
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and not torch.cuda.is_bf16_supported())
    for batch in loader:
        x, mask, y, w = (batch[k].to(device) for k in ("x", "mask", "y", "w"))
        if not w.any():
            continue
        optimizer.zero_grad(set_to_none=True)
        with amp_context(device):
            loss = weighted_bce(head(x, mask), y, w)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(head.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total += float(loss.detach()) * len(x)
        count += len(x)
    if count == 0:
        raise ValueError("Training split contains no usable labels for this member")
    return total / count


def train_head(model, features, masks, train_indices, y, weights, groups, gold, cfg, device, fold):
    inner_fold = balanced_group_folds(y[train_indices], gold[train_indices], groups[train_indices],
                                     min(5, len(np.unique(groups[train_indices]))), cfg.seed + fold)
    inner_train, inner_val = train_indices[inner_fold != 0], train_indices[inner_fold == 0]
    if len(inner_train) == 0 or len(inner_val) == 0:
        raise ValueError("Insufficient groups for independent inner epoch selection")
    head_seed = cfg.seed + 1000 + fold
    seed_everything(head_seed)
    model.head = FindingHead(model.backbone.num_features, cfg).to(device)
    optimizer = torch.optim.AdamW(model.head.parameters(), lr=cfg.head_lr, weight_decay=.01)
    tr_loader = data_loader(FeatureDataset(features, masks, inner_train, y, weights), cfg, cfg.head_batch, True, head_seed)
    va_loader = data_loader(FeatureDataset(features, masks, inner_val, y, weights), cfg, cfg.head_batch)
    best, best_epoch, stale = math.inf, 1, 0
    for epoch in range(1, cfg.head_epochs + 1):
        training_loss = head_epoch(model.head, tr_loader, optimizer, device)
        validation_loss = evaluate_loss(model.head, va_loader, device)
        print(f"Fold {fold}, inner epoch {epoch}: train BCE {training_loss:.5f}, val BCE {validation_loss:.5f}")
        if validation_loss < best - 1e-5:
            best, best_epoch, stale = validation_loss, epoch, 0
        else:
            stale += 1
            if stale >= cfg.patience:
                break
    # Refit from the same initialization on all outer-training groups for chosen epochs.
    seed_everything(head_seed)
    model.head = FindingHead(model.backbone.num_features, cfg).to(device)
    optimizer = torch.optim.AdamW(model.head.parameters(), lr=cfg.head_lr, weight_decay=.01)
    loader = data_loader(FeatureDataset(features, masks, train_indices, y, weights), cfg, cfg.head_batch, True, head_seed)
    for epoch in range(best_epoch):
        head_epoch(model.head, loader, optimizer, device)
    return {"selected_head_epochs": best_epoch, "inner_validation_bce": best,
            "outer_refit_studies": len(train_indices)}


def augment_study(images):
    b, s, k, h, w = images.shape
    angles = (torch.rand(b, device=images.device) - .5) * math.pi / 18
    scales = 1 + (torch.rand(b, device=images.device) - .5) * .10
    theta = images.new_zeros((b, 2, 3), dtype=torch.float32)
    theta[:, 0, 0], theta[:, 1, 1] = angles.cos() / scales, angles.cos() / scales
    theta[:, 0, 1], theta[:, 1, 0] = -angles.sin() / scales, angles.sin() / scales
    theta[:, :, 2] = (torch.rand(b, 2, device=images.device) - .5) * .06
    theta = theta.repeat_interleave(s * k, 0)
    x = images.reshape(b * s * k, 1, h, w).float()
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, align_corners=False)
    brightness = 1 + (torch.rand(b, device=images.device) - .5) * .10
    return (x.reshape(b, s, k, h, w) * brightness[:, None, None, None, None]).clamp(0, 255)


def finetune(model, folder, ids, indices, y, weights, cfg, device, seed):
    if cfg.finetune_epochs == 0:
        return
    seed_everything(seed)
    model.unfreeze(cfg.unfreeze_blocks)
    model.encode_batch = min(cfg.encode_batch, 8)
    loader = data_loader(ImageDataset(folder, np.asarray(ids)[indices], y[indices], weights[indices]),
                         cfg, cfg.image_batch, True, seed)
    optimizer = torch.optim.AdamW([
        {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": cfg.backbone_lr},
        {"params": model.head.parameters(), "lr": cfg.head_lr / 3}], weight_decay=.01)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and not torch.cuda.is_bf16_supported())
    for epoch in range(cfg.finetune_epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total = 0.0
        for step, batch in enumerate(loader):
            x, mask, target, weight = (batch[k].to(device) for k in ("x", "mask", "y", "w"))
            window_start = (step // cfg.accumulation) * cfg.accumulation
            divisor = min(cfg.accumulation, len(loader) - window_start)
            with amp_context(device):
                loss = weighted_bce(model(augment_study(x), mask), target, weight)
            scaler.scale(loss / divisor).backward()
            if (step + 1) % cfg.accumulation == 0 or step + 1 == len(loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            total += float(loss.detach())
        print(f"Fixed fine-tuning epoch {epoch + 1}/{cfg.finetune_epochs}: BCE {total / len(loader):.5f}")
    model.encode_batch = cfg.encode_batch
    model.eval()


def predict_features(head, features, masks, indices, cfg, device):
    loader = data_loader(FeatureDataset(features, masks, indices), cfg, cfg.head_batch)
    values = []
    head.eval()
    with torch.inference_mode():
        for batch in loader:
            with amp_context(device):
                logits = head(batch["x"].to(device), batch["mask"].to(device))
            values.append(logits.float().sigmoid().cpu().numpy())
    result = np.concatenate(values)
    if not np.isfinite(result).all():
        raise ValueError("Nonfinite predictions")
    return result


def predict_images(model, folder, ids, cfg, device):
    loader = data_loader(ImageDataset(folder, ids), cfg, cfg.image_batch)
    values = []
    model.eval()
    with torch.inference_mode():
        for batch in loader:
            with amp_context(device):
                logits = model(batch["x"].to(device), batch["mask"].to(device))
            values.append(logits.float().sigmoid().cpu().numpy())
    result = np.concatenate(values)
    if not np.isfinite(result).all():
        raise ValueError("Nonfinite predictions")
    return result




## Checkpoint contracts, OOF diagnostics, and strict rank blending

In [ ]:
def implementation_hash():
    if "_IMPLEMENTATION_SHA" in globals():
        return globals()["_IMPLEMENTATION_SHA"]
    return digest([file_sha(__file__), file_sha(Path(__file__).with_name("report_rules.py"))])


def rank_predictions(values):
    array = np.asarray(values, float)
    if array.ndim != 2 or not np.isfinite(array).all():
        raise ValueError("Rank blending requires a finite two-dimensional prediction matrix")
    return pd.DataFrame(array).rank(method="average", pct=True).to_numpy(np.float32)


def target_weights(member):
    if member.weight <= 0 or not np.isfinite(member.weight):
        raise ValueError("Ensemble weights must be positive and finite")
    if not member.targets or set(member.targets) - set(TARGETS):
        raise ValueError("Specialists must name official findings")
    return np.array([member.weight if t in member.targets else 0.0 for t in TARGETS], np.float32)


def save_checkpoint(path, model, payload):
    temp = path.with_suffix(".tmp.pt")
    torch.save({**payload, "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()}}, temp)
    os.replace(temp, path)


def load_checkpoint(path, expected_sha=None):
    if expected_sha is not None and file_sha(path) != expected_sha:
        raise ValueError(f"Checkpoint checksum mismatch: {path.name}")
    payload = torch.load(path, map_location="cpu", weights_only=True)
    if payload.get("version") != VERSION or payload.get("targets") != TARGETS:
        raise ValueError("Checkpoint is not from this pipeline or its target order differs")
    if payload.get("label_source") != "deterministic_e13_rules_and_official_gold_only":
        raise ValueError("Checkpoint label provenance is incompatible with the no-LLM requirement")
    return payload


def checkpoint_model(payload, device):
    cfg, member = Config(**payload["config"]), Member(**payload["member"])
    model = StudyModel(member, cfg, pretrained=False)  # no network access in inference
    model.load_state_dict(payload["state_dict"], strict=True)
    return model.to(device).eval()


def validate_config(cfg, members):
    if cfg.mode not in ("train", "infer", "train_infer"):
        raise ValueError("mode must be train, infer or train_infer")
    if cfg.folds < 2 or not cfg.train_folds or set(cfg.train_folds) - set(range(cfg.folds)):
        raise ValueError("Invalid fold selection")
    if cfg.head_epochs < 1 or cfg.finetune_epochs < 0 or cfg.slices < 1 or cfg.image_size < 16:
        raise ValueError("Invalid training/image configuration")
    if cfg.image_batch < 1 or cfg.head_batch < 1 or cfg.encode_batch < 1 or cfg.accumulation < 1:
        raise ValueError("Batch sizes and accumulation must be positive")
    if not 0 <= cfg.slice_band[0] < cfg.slice_band[1] <= 1 or cfg.crop_mm <= 0:
        raise ValueError("Invalid physical field of view or slice band")
    if len({m.name for m in members}) != len(members) or not members:
        raise ValueError("Member names must be distinct and at least one member is required")
    coverage = np.zeros(len(TARGETS))
    for m in members:
        if not m.name.replace("_", "").isalnum():
            raise ValueError("Member names must use letters, digits, and underscores")
        coverage += target_weights(m)
        patch = 14 if "patch14" in m.backbone else 16 if "patch16" in m.backbone else 1
        if cfg.image_size % patch:
            raise ValueError(f"{m.name}: image_size must be divisible by patch size {patch}")
        if not m.provenance.strip():
            raise ValueError("Document the origin of every image foundation weight")
    if cfg.mode != "infer" and (coverage == 0).any():
        raise ValueError("Training ensemble must cover every official finding")


def train_artifacts(root, cfg, members, device):
    out = Path(cfg.output_dir)
    train = read_table(root, "train.csv")
    test = read_table(root, "test.csv")
    validate_studies(train, "train")
    validate_studies(test, "test")
    if set(train[UID]) & set(test[UID]):
        raise ValueError("Training and test study IDs overlap")
    y, weights, gold, label_audit = make_targets(train, cfg)
    groups = report_patient_groups(train)
    folds = balanced_group_folds(y, gold, groups, cfg.folds, cfg.seed)
    train_config = dataclasses.asdict(cfg)
    for key in ("mode", "data_root", "output_dir", "checkpoint_dir", "train_folds", "resume", "workers", "header_workers", "require_gpu"):
        train_config.pop(key)
    signature = digest({"version": VERSION, "implementation": implementation_hash(),
                        "train_csv": file_sha(root / "train.csv"), "config": train_config,
                        "members": [dataclasses.asdict(m) for m in members], "folds": folds.tolist()})
    signature_file = out / "training_signature.json"
    if signature_file.exists() and json.loads(signature_file.read_text())["signature"] != signature:
        raise ValueError("This output directory belongs to a different experiment; choose a new output_dir")
    atomic_json(signature_file, {"signature": signature})
    label_audit.to_csv(out / "label_audit.csv", index=False)
    pd.DataFrame({UID: train[UID], "group": groups, "fold": folds}).to_csv(out / "folds.csv", index=False)
    label_frame = pd.DataFrame(y, columns=TARGETS)
    label_frame.insert(0, UID, train[UID])
    for j, target in enumerate(TARGETS):
        label_frame[target + "__weight"] = weights[:, j]
        label_frame[target + "__official"] = gold[:, j]
    label_frame.to_csv(out / "training_targets.csv", index=False)
    print(label_audit.to_string(index=False))
    print("Group sizes:", pd.Series(folds).value_counts().sort_index().to_dict())
    if not any(c in train for c in ("PatientID", "PatientInstanceUID", "patient_id")):
        print("No patient identifier provided: grouping protects duplicate reports; patient-disjointness cannot be established.")
    folder, receipt = build_image_cache(root, "train", train, cfg)
    files, member_oof, diagnostics = [], [], {}
    for member in members:
        member_weight = weights * (target_weights(member) > 0)[None]
        model, features, masks = None, None, None
        oof = np.full(y.shape, np.nan, np.float32)
        fold_scores = []
        for fold in range(cfg.folds):
            path = out / f"{member.name}_fold{fold}.pt"
            train_idx, val_idx = np.flatnonzero(folds != fold), np.flatnonzero(folds == fold)
            if path.exists() and cfg.resume:
                payload = load_checkpoint(path)
                if payload["training_signature"] != signature or payload["validation_ids"] != train.iloc[val_idx][UID].tolist():
                    raise ValueError("Existing fold checkpoint has incompatible data/configuration")
                predictions = payload["validation_predictions"].numpy()
                print(f"Reuse completed {member.name} fold {fold}")
            elif fold in cfg.train_folds:
                if features is None:
                    seed_everything(cfg.seed)
                    model = load_foundation(member, cfg, device)
                    features, masks = extract_feature_cache(model, folder, train[UID], receipt, cfg, device)
                elif cfg.finetune_epochs:
                    # Start EVERY fold from the original foundation, never a previous fold's model.
                    del model
                    gc.collect()
                    if device.type == "cuda":
                        torch.cuda.empty_cache()
                    seed_everything(cfg.seed)
                    model = load_foundation(member, cfg, device)
                selection = train_head(model, features, masks, train_idx, y, member_weight,
                                       groups, gold, cfg, device, fold)
                finetune(model, folder, train[UID], train_idx, y, member_weight,
                         cfg, device, cfg.seed + 2000 + fold)
                if cfg.finetune_epochs:
                    predictions = predict_images(model, folder, train.iloc[val_idx][UID], cfg, device)
                else:
                    predictions = predict_features(model.head, features, masks, val_idx, cfg, device)
                save_checkpoint(path, model, {"version": VERSION, "targets": TARGETS,
                    "label_source": "deterministic_e13_rules_and_official_gold_only",
                    "config": dataclasses.asdict(cfg), "member": dataclasses.asdict(member),
                    "training_signature": signature, "implementation_sha256": implementation_hash(),
                    "fold": fold, "selection": selection, "backbone_sha256": backbone_sha(model),
                    "validation_ids": train.iloc[val_idx][UID].tolist(),
                    "validation_predictions": torch.from_numpy(predictions),
                    "engineering_only_random_init": cfg.allow_random_init})
            else:
                continue
            if predictions.shape != (len(val_idx), len(TARGETS)) or not np.isfinite(predictions).all():
                raise ValueError("Invalid checkpoint OOF predictions")
            oof[val_idx] = predictions
            score = auc_report(y[val_idx], predictions, gold[val_idx])
            fold_scores.append({"fold": fold, **score})
            print(f"{member.name} fold {fold} gold diagnostics: {score['macro_over_defined_targets']} ({score['defined_targets']}/12 defined)")
            files.append({"filename": path.name, "sha256": file_sha(path), "member": member.name,
                          "fold": fold, "weights": target_weights(member).tolist()})
        frame = pd.DataFrame(oof, columns=TARGETS)
        frame.insert(0, "fold", folds)
        frame.insert(0, UID, train[UID])
        frame.to_csv(out / f"oof_{member.name}.csv", index=False)
        # Ranks are computed within each outer fold, matching the deployment population contract.
        oof_ranks = np.full_like(oof, np.nan)
        for fold in range(cfg.folds):
            mask = (folds == fold) & np.isfinite(oof).all(1)
            if mask.any():
                oof_ranks[mask] = rank_predictions(oof[mask])
        member_oof.append((oof_ranks, target_weights(member)))
        diagnostics[member.name] = {"gold_oof_rank_diagnostic": auc_report(y, oof_ranks, gold),
                                    "fold_gold": fold_scores}
        del model, features, masks
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
    complete = len(files) == cfg.folds * len(members)
    manifest = {"version": VERSION, "targets": TARGETS, "complete": complete,
                "preprocessing": preprocessing(cfg), "training_signature": signature,
                "files": files, "member_names": [m.name for m in members], "folds": cfg.folds,
                "label_source": "deterministic_e13_rules_and_official_gold_only",
                "engineering_only_random_init": cfg.allow_random_init,
                "implementation_sha256": implementation_hash(),
                "versions": {"torch": torch.__version__, "timm": timm.__version__,
                             "numpy": np.__version__, "pydicom": pydicom.__version__}}
    atomic_json(out / "manifest.json", manifest)
    if complete:
        numerator, denominator = np.zeros_like(y), np.zeros(len(TARGETS))
        for predictions, weight in member_oof:
            numerator += predictions * weight[None]
            denominator += weight
        ensemble = numerator / denominator[None]
        diagnostics["ensemble"] = {"gold_oof_rank_diagnostic": auc_report(y, ensemble, gold)}
        pd.DataFrame(ensemble, columns=TARGETS).assign(**{UID: train[UID].tolist(), "fold": folds}).to_csv(out / "oof_ensemble.csv", index=False)
    diagnostics["interpretation"] = (
        "OOF diagnostics use only official expert annotations, mask missing labels, and report undefined AUCs. "
        "Inner groups select head epochs; outer folds are held out through refit and fixed fine-tuning. "
        "Pooled fold ranks are diagnostics, not a Kaggle score. No ensemble weights are fitted on outer OOF or test labels.")
    atomic_json(out / "validation.json", diagnostics)
    print("Saved", out / "manifest.json", "complete=", complete)
    return out


def validate_submission(test, values, path, sample=None):
    validate_studies(test, "test")
    if np.asarray(values).shape != (len(test), len(TARGETS)):
        raise ValueError("Prediction count or target count differs from test.csv")
    values = np.asarray(values, float)
    if not np.isfinite(values).all() or (values < 0).any() or (values > 1).any():
        raise ValueError("Submission predictions must be finite numbers in [0,1]")
    columns = [UID, *TARGETS]
    if sample is not None and list(sample.columns) != columns:
        raise ValueError("sample_submission.csv has an unexpected column contract")
    submission = pd.DataFrame(values, columns=TARGETS)
    submission.insert(0, UID, test[UID].tolist())
    temp = Path(path).with_suffix(".tmp.csv")
    submission.to_csv(temp, index=False)
    check = pd.read_csv(temp, dtype={UID: str})
    if check[UID].tolist() != test[UID].tolist() or check.columns.tolist() != columns:
        raise ValueError("Submission failed round-trip identifier/schema check")
    os.replace(temp, path)
    return submission


def infer_artifacts(root, cfg, checkpoint_dir, device):
    source = Path(checkpoint_dir)
    manifest = json.loads((source / "manifest.json").read_text())
    if manifest.get("version") != VERSION or manifest.get("targets") != TARGETS or not manifest.get("complete"):
        raise ValueError("Attach a complete manifest plus all trained fold checkpoints from this notebook")
    if manifest.get("engineering_only_random_init") and not cfg.allow_random_init:
        raise ValueError("Synthetic random-weight artifacts are not valid competition models")
    if manifest.get("label_source") != "deterministic_e13_rules_and_official_gold_only":
        raise ValueError("Ensemble manifest violates the label provenance contract")
    if manifest.get("implementation_sha256") != implementation_hash():
        raise ValueError("Model artifacts require the same generated pipeline version used to train them")
    expected = {(name, fold) for name in manifest["member_names"] for fold in range(manifest["folds"])}
    observed = [(r["member"], r["fold"]) for r in manifest["files"]]
    if len(set(observed)) != len(observed) or set(observed) != expected:
        raise ValueError("Manifest contains missing, duplicate, or unexpected ensemble members")
    restored_cfg = dataclasses.replace(cfg, **manifest["preprocessing"])
    test = read_table(root, "test.csv")
    validate_studies(test, "test")
    folder, receipt = build_image_cache(root, "test", test, restored_cfg)
    total, denominator = np.zeros((len(test), len(TARGETS)), np.float64), np.zeros(len(TARGETS))
    for entry in manifest["files"]:
        if Path(entry["filename"]).name != entry["filename"]:
            raise ValueError("Checkpoint filenames must be basenames inside the attached dataset")
        payload = load_checkpoint(source / entry["filename"], entry["sha256"])
        if payload["training_signature"] != manifest["training_signature"]:
            raise ValueError("Mixed experiments in checkpoint ensemble")
        member = Member(**payload["member"])
        if member.name != entry["member"] or payload["fold"] != entry["fold"]:
            raise ValueError("Manifest/checkpoint member mismatch")
        if payload["implementation_sha256"] != manifest["implementation_sha256"]:
            raise ValueError("Manifest/checkpoint implementation mismatch")
        if digest(preprocessing(Config(**payload["config"]))) != digest(manifest["preprocessing"]):
            raise ValueError("Mixed image preprocessing contracts")
        weights = target_weights(member)
        if not np.array_equal(weights, np.array(entry["weights"], np.float32)):
            raise ValueError("Manifest/checkpoint blend weights differ")
        model = checkpoint_model(payload, device)
        model.encode_batch = cfg.encode_batch
        # Frozen folds reuse the same features; tuned folds each get a distinct content key.
        features, masks = extract_feature_cache(model, folder, test[UID], receipt, restored_cfg, device)
        predictions = predict_features(model.head, features, masks, np.arange(len(test)), restored_cfg, device)
        total += rank_predictions(predictions) * weights[None]
        denominator += weights
        print(f"Scored {entry['member']} fold {entry['fold']}: {len(test)} test studies")
        del model, features, masks, payload
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
    if (denominator <= 0).any():
        raise ValueError("Ensemble does not cover all twelve findings")
    values = total / denominator[None]
    path = Path(cfg.output_dir).parent / "submission.csv"
    sample = read_table(root, "sample_submission.csv") if (root / "sample_submission.csv").exists() else None
    submission = validate_submission(test, values, path, sample)
    atomic_json(Path(cfg.output_dir) / "submission_audit.json", {
        "version": VERSION, "test_studies": len(test), "members_executed": len(manifest["files"]),
        "output_sha256": file_sha(path), "test_csv_sha256": file_sha(root / "test.csv"),
        "training_signature": manifest["training_signature"],
        "label_source": manifest["label_source"], "constant_targets": [
            t for j, t in enumerate(TARGETS) if len(np.unique(values[:, j])) <= 1]})
    print("Validated submission:", path, submission.shape)
    return submission




## Execute a selected workflow

In [ ]:
def run(cfg, members):
    start = time.monotonic()
    validate_config(cfg, members)
    seed_everything(cfg.seed)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    if cfg.require_gpu and device.type != "cuda":
        raise RuntimeError("Enable a Kaggle GPU accelerator; CPU is supported only for explicit small engineering checks")
    if cfg.mode == "infer" and not cfg.checkpoint_dir:
        raise ValueError("Set checkpoint_dir to the attached training output containing manifest.json")
    out = Path(cfg.output_dir)
    out.mkdir(parents=True, exist_ok=True)
    root = find_data_root(cfg)
    print(f"Device={device}; data={root}; mode={cfg.mode}; seed={cfg.seed}")
    if device.type == "cuda":
        print(torch.cuda.get_device_name(0), f"VRAM={torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
    print(f"Image cache upper bound per 1,000 studies: {1000 * 6 * cfg.slices * cfg.image_size**2 / 2**30:.2f} GiB")
    checkpoint_dir = cfg.checkpoint_dir
    if cfg.mode in ("train", "train_infer"):
        checkpoint_dir = str(train_artifacts(root, cfg, members, device))
    result = infer_artifacts(root, cfg, checkpoint_dir, device) if cfg.mode in ("infer", "train_infer") else None
    print(f"Completed in {(time.monotonic() - start) / 60:.1f} minutes")
    return result


## Fast engineering checks

In [ ]:
# Small contract checks run before expensive dataset work.
example = extract("Complete ACL rupture. No joint effusion. Large Baker cyst.")
assert example["ACL"] > 0.5 and example["Effusion"] < 0.5
assert example["Baker's"] > 0.5
assert list(TARGETS) == ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
                        "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis",
                        "Baker's", "Contusion", "Fracture"]
tiny_cfg = dataclasses.replace(CFG, slices=2, hidden=16, dropout=0.0)
tiny_head = FindingHead(8, tiny_cfg).eval()
tiny_features = torch.randn(2, 6, 2, 8)
tiny_mask = torch.ones(2, 6, 2, dtype=torch.bool)
tiny_mask[:, -1] = False
with torch.inference_mode():
    before = tiny_head(tiny_features, tiny_mask)
    tiny_features[:, -1] = 1000  # missing series must not change the prediction
    after = tiny_head(tiny_features, tiny_mask)
assert before.shape == (2, 12) and torch.isfinite(before).all()
assert torch.allclose(before, after, atol=1e-6)
del tiny_head, tiny_features, tiny_mask
print("Report polarity, target order, and missing-series attention contracts passed")


## Run

This executes the complete selected workflow. Errors stop the notebook; inspect the exception and audit files.

In [ ]:
RESULT = run(CFG, MEMBERS)
if RESULT is not None:
    display(RESULT.head())

## Outputs and troubleshooting

Training writes checkpoints, checksum manifest, `training_targets.csv`, `label_audit.csv`,
`folds.csv`, each `oof_*.csv`, and gold-only `validation.json`. Image receipts record
sequence choices, unknown laterality, metadata disagreement, and individual decode errors.
Infer/train_infer write `submission.csv` and `submission_audit.json` only after all required
fold members execute and schema/IDs/ranges pass. There is no constant fallback submission.

* **No weights:** enable Internet during training for DINOv2 foundation download, or attach
  a compatible checksum-pinned foundation file. A trained checkpoint contains its complete
  backbone and never downloads it during inference. Public competition heads are unsupported.
* **GPU out of memory:** lower `encode_batch`, keep `image_batch=1`, and use the frozen
  baseline before fine-tuning. Changing an experiment's training configuration requires a
  fresh output directory. Fine-tuning uses AMP, gradient accumulation, clipped gradients,
  and ViT gradient checkpointing; it deliberately keeps BatchNorm statistics fixed.
* **Long run:** `train_folds` controls completed-fold work. Resume validates the experiment
  signature. Partial folds are research outputs, not a complete inference ensemble.
* **DICOM decoding:** uncompressed data works with pydicom/Pillow. For JPEG-LS/JPEG2000,
  use Kaggle's installed decoder or attach compatible `pylibjpeg` plugin wheels. Decoder
  failures are audited and masked; a study with no usable image stops the run. No verification
  or assertion is disabled to produce a file.
* **Cache budget:** the default uncompressed upper bound is 2.24 GiB per 1,000 studies;
  image files are compressed and each feature cache is float16 on disk. Fine-tuned folds
  produce distinct test feature caches. Check free disk and run one member at a time if
  you need to manage your Kaggle output quota. Full test runtime must be measured on Kaggle.
* **Small gold set:** undefined AUCs are reported explicitly. A diagnostic average over
  fewer than twelve targets is not the complete official metric. Rules and ensemble
  weights should be revised using additional independent annotation/validation rather
  than repeated public leaderboard probing.

